# Biohub Targeted Postprocess Ablation 51

Runs multiple DeepCenter postprocess variants from the saved 199-sample checkpoint GEFFs. It does **not** rerun model inference. Default target set is all 39 problem samples plus 12 high-score guard samples. Results are written under `reports/`.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import sys
import zipfile
import time

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

PROJECT_DIR = Path('/content/drive/MyDrive/Biohub - Cell Tracking During Development')
BASE_DIR = PROJECT_DIR / 'data'
REPORT_DIR = PROJECT_DIR / 'reports'
ARCHIVE_DIR = PROJECT_DIR / 'external_archives'
ARTIFACT_DIR = PROJECT_DIR / 'artifacts'
CHECKPOINT_DIR = REPORT_DIR / 'deepcenter_full199_prediction_geff_checkpoint'
TARGET_SETS_CSV = REPORT_DIR / 'targeted_validation_sets.csv'

# Heavy temporary build files stay on /content. Final CSVs/scores go to Drive reports/.
WORK_DIR = Path('/content/biohub_targeted_ablation_work')
LOCAL_TEST_DIR = Path('/content/biohub_targeted_ablation/test')

# Default: all problem samples + enough high-score guards.
# Problem samples are the union of bottom24, low-recall, motion/gap-risk, and branch/div-risk flags.
# Current source CSV gives 39 bad/problem samples + 12 guard samples = 51 total.
TARGET_SET_NAME = 'all_bad_plus_guard'

# Run order. Keep baseline first so every variant is compared to the exact same rebuild path.
VARIANTS_TO_RUN = [
    'baseline_rebuild',
    'safe_div_conservative_v2',
    'safe_div_conservative_v3',
    'motion_gap_conservative_v1',
    'motion_gap_conservative_v2',
    'short_track_len5_v1',
    'linefit_low_weight_v1',
    'edge_max_12_5_v1',
    'combo_safe_motion_v1',
    'combo_safe_motion_short_v1',
]

# Postprocess-only variants. These do not rerun the expensive model inference; they rebuild CSVs from saved GEFF predictions.
TUNING_VARIANTS = {
    'baseline_rebuild': {},
    # Diagnostic already tested; kept here for reference but not in default run list.
    'no_safe_divisions_diagnostic': {
        'BIOHUB_OUTPUT_SAFE_DIVISIONS': '0',
    },
    # Keep safe divisions, but make them less eager. Full off was bad; this is the softer hypothesis.
    'safe_div_conservative_v2': {
        'BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP': '0.0015',
        'BIOHUB_SAFE_DIV_FRAME_FRAC_CAP': '0.003',
        'BIOHUB_SAFE_DIV_MAX_UM': '3.8',
        'BIOHUB_SAFE_DIV_SISTER_MAX_UM': '6.0',
        'BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM': '6.2',
    },
    'safe_div_conservative_v3': {
        'BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP': '0.0010',
        'BIOHUB_SAFE_DIV_FRAME_FRAC_CAP': '0.002',
        'BIOHUB_SAFE_DIV_MAX_UM': '3.4',
        'BIOHUB_SAFE_DIV_SISTER_MAX_UM': '5.5',
        'BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM': '5.8',
    },
    # Worst-sample analysis showed motion_relink_relaxed_edges and gap_added_nodes_frac are strongly negative.
    'motion_gap_conservative_v1': {
        'BIOHUB_MOTION_RELINK_RELAXED_UM': '8.0',
        'BIOHUB_MOTION_RELINK_TIGHT_UM': '5.5',
        'BIOHUB_GAP_CLOSE_UM': '5.0',
        'BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC': '0.025',
        'BIOHUB_GAP_CLOSE_MAX_ADDED_ABS': '1000',
    },
    'motion_gap_conservative_v2': {
        'BIOHUB_MOTION_RELINK_RELAXED_UM': '7.0',
        'BIOHUB_MOTION_RELINK_TIGHT_UM': '5.0',
        'BIOHUB_GAP_CLOSE_UM': '4.5',
        'BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC': '0.015',
        'BIOHUB_GAP_CLOSE_MAX_ADDED_ABS': '700',
    },
    # If short-track filtering deletes sparse GT fragments, len 5 can recover edges.
    'short_track_len5_v1': {
        'BIOHUB_OUTPUT_MIN_TRACK_LEN': '5',
    },
    # match_distance_p95 is the strongest negative signal; reduce smoothing/linefit pull.
    'linefit_low_weight_v1': {
        'BIOHUB_OUTPUT_LINEFIT_WEIGHT': '0.35',
        'BIOHUB_OUTPUT_LINEFIT_WINDOW': '1',
    },
    # Cap very long final edges. Can help if long relaxed links are noisy; can hurt true long moves.
    'edge_max_12_5_v1': {
        'BIOHUB_OUTPUT_EDGE_MAX_UM': '12.5',
    },
    'combo_safe_motion_v1': {
        'BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP': '0.0015',
        'BIOHUB_SAFE_DIV_FRAME_FRAC_CAP': '0.003',
        'BIOHUB_SAFE_DIV_MAX_UM': '3.8',
        'BIOHUB_SAFE_DIV_SISTER_MAX_UM': '6.0',
        'BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM': '6.2',
        'BIOHUB_MOTION_RELINK_RELAXED_UM': '8.0',
        'BIOHUB_MOTION_RELINK_TIGHT_UM': '5.5',
        'BIOHUB_GAP_CLOSE_UM': '5.0',
        'BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC': '0.025',
        'BIOHUB_GAP_CLOSE_MAX_ADDED_ABS': '1000',
    },
    'combo_safe_motion_short_v1': {
        'BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP': '0.0015',
        'BIOHUB_SAFE_DIV_FRAME_FRAC_CAP': '0.003',
        'BIOHUB_SAFE_DIV_MAX_UM': '3.8',
        'BIOHUB_SAFE_DIV_SISTER_MAX_UM': '6.0',
        'BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM': '6.2',
        'BIOHUB_MOTION_RELINK_RELAXED_UM': '8.0',
        'BIOHUB_MOTION_RELINK_TIGHT_UM': '5.5',
        'BIOHUB_GAP_CLOSE_UM': '5.0',
        'BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC': '0.025',
        'BIOHUB_GAP_CLOSE_MAX_ADDED_ABS': '1000',
        'BIOHUB_OUTPUT_MIN_TRACK_LEN': '5',
    },
}

VARIANT_ENV_KEYS = sorted({key for params in TUNING_VARIANTS.values() for key in params})

for out_dir in [REPORT_DIR, ARCHIVE_DIR, ARTIFACT_DIR, WORK_DIR, LOCAL_TEST_DIR]:
    out_dir.mkdir(parents=True, exist_ok=True)

if not BASE_DIR.exists():
    raise FileNotFoundError(f'BASE_DIR is missing: {BASE_DIR}')
if not CHECKPOINT_DIR.exists():
    raise FileNotFoundError(f'Prediction checkpoint is missing: {CHECKPOINT_DIR}')

# Artifact zip helpers.
def zip_contains(zip_path: Path, needle: str) -> bool:
    try:
        with zipfile.ZipFile(zip_path) as zf:
            return any(name.endswith(needle) or needle in name for name in zf.namelist())
    except Exception:
        return False


def find_artifact_zips():
    zips = []
    for root in [ARCHIVE_DIR, PROJECT_DIR, Path('/content')]:
        if root.exists():
            zips.extend(sorted(root.glob('*.zip')))
    support = None
    center = None
    figures = None
    for zp in zips:
        if support is None and zip_contains(zp, 'repo/scripts/predict_unet_transformer.py'):
            support = zp
        if center is None and zip_contains(zp, 'weights/full_frame_center/checkpoint_last.pt'):
            center = zp
        if figures is None and zip_contains(zp, 'biohub_lineage.png'):
            figures = zp
    return support, center, figures, zips


def extract_if_needed(zip_path: Path | None, target_dir: Path, marker: str):
    if (target_dir / marker).exists():
        print('already extracted:', target_dir)
        return True
    if zip_path is None:
        print('missing zip for', target_dir.name)
        return False
    target_dir.mkdir(parents=True, exist_ok=True)
    print('extracting', zip_path.name, '->', target_dir)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(target_dir)
    print('done')
    return (target_dir / marker).exists()

support_zip, center_zip, figures_zip, all_zips = find_artifact_zips()
print('found zips:', [p.name for p in all_zips])
print('support_zip:', support_zip)
print('center_zip:', center_zip)

SUPPORT_DIR = ARTIFACT_DIR / 'biohub-tracking-support-pack-50ep-v1'
CENTER_DIR = ARTIFACT_DIR / 'biohub-deepcenter-unet3d-center-prior-v1'
assert extract_if_needed(support_zip, SUPPORT_DIR, 'repo/scripts/predict_unet_transformer.py'), 'support artifact missing/incomplete'
assert extract_if_needed(center_zip, CENTER_DIR, 'weights/full_frame_center/checkpoint_last.pt'), 'center artifact missing/incomplete'

# Build target CSV on the fly if it has not been copied to Drive yet.
def ensure_target_sets_csv():
    if TARGET_SETS_CSV.exists():
        return
    import pandas as pd
    scores_path = REPORT_DIR / 'deepcenter_full199_local_validation_scores.csv'
    stats_path = REPORT_DIR / 'deepcenter_full199_local_validation_run_stats.csv'
    if not scores_path.exists() or not stats_path.exists():
        raise FileNotFoundError('Need full199 score/run_stats reports to construct targeted sets.')
    scores = pd.read_csv(scores_path)
    stats = pd.read_csv(stats_path)
    score_col = 'edge_jaccard_adjusted_approx'
    merged = scores.merge(stats, left_on='sample_id', right_on='dataset', how='left', suffixes=('', '_stat'))
    bottom24 = set(merged.nsmallest(24, score_col)['sample_id'])
    low_recall = set(merged.nsmallest(60, score_col).query('node_recall_sparse_gt < 0.96').sort_values('node_recall_sparse_gt').head(12)['sample_id'])
    temp = merged.nsmallest(60, score_col).copy()
    temp['stress_rank'] = temp[['match_distance_p95_um', 'motion_relink_relaxed_edges', 'gap_added_nodes_frac', 'short_track_nodes_removed']].rank(pct=True).mean(axis=1)
    motion_gap = set(temp.sort_values('stress_rank', ascending=False).head(16)['sample_id'])
    branch_div = set(merged.nsmallest(80, score_col).sort_values('division_like_sources', ascending=False).head(16)['sample_id'])
    high = merged[merged[score_col] >= 0.985].copy()
    if len(high) < 12:
        high = merged.nlargest(40, score_col).copy()
    guard = []
    for embryo in ['44b6', '6bba']:
        pool = high[high['embryo_id'].eq(embryo)].copy()
        for q in [0.15, 0.5, 0.85]:
            if len(pool):
                target = pool['pred_nodes'].quantile(q)
                guard.append(pool.iloc[(pool['pred_nodes'] - target).abs().argsort()[:1]]['sample_id'].iloc[0])
    for sid in merged.sort_values(score_col, ascending=False)['sample_id']:
        if len(dict.fromkeys(guard)) >= 12:
            break
        guard.append(sid)
    guard = list(dict.fromkeys(guard))[:12]
    all_ids = sorted(bottom24 | low_recall | motion_gap | branch_div | set(guard))
    out = merged[merged.sample_id.isin(all_ids)].copy()
    out['bottom24_edge_rescue'] = out.sample_id.isin(bottom24).astype(int)
    out['low_recall_rescue'] = out.sample_id.isin(low_recall).astype(int)
    out['motion_gap_conservative'] = out.sample_id.isin(motion_gap).astype(int)
    out['branch_div_conservative'] = out.sample_id.isin(branch_div).astype(int)
    out['guard_high_score'] = out.sample_id.isin(guard).astype(int)
    out['balanced_target_guard'] = 1
    out.to_csv(TARGET_SETS_CSV, index=False)
    print('created:', TARGET_SETS_CSV, 'rows:', len(out))

ensure_target_sets_csv()

import pandas as pd
target_df = pd.read_csv(TARGET_SETS_CSV)
if TARGET_SET_NAME == 'bottom24_plus_guard':
    selected = target_df[(target_df['bottom24_edge_rescue'].eq(1)) | (target_df['guard_high_score'].eq(1))]
elif TARGET_SET_NAME == 'bottom24_lowrecall_plus_guard':
    selected = target_df[(target_df['bottom24_edge_rescue'].eq(1)) | (target_df['low_recall_rescue'].eq(1)) | (target_df['guard_high_score'].eq(1))]
elif TARGET_SET_NAME == 'all_bad_plus_guard':
    bad_mask = (target_df['bottom24_edge_rescue'].eq(1) |
                target_df['low_recall_rescue'].eq(1) |
                target_df['motion_gap_conservative'].eq(1) |
                target_df['branch_div_conservative'].eq(1))
    selected = target_df[bad_mask | target_df['guard_high_score'].eq(1)]
elif TARGET_SET_NAME in target_df.columns:
    selected = target_df[target_df[TARGET_SET_NAME].eq(1)]
else:
    raise ValueError(f'Unknown TARGET_SET_NAME={TARGET_SET_NAME}. Available columns: {target_df.columns.tolist()}')

sample_ids = selected['sample_id'].astype(str).drop_duplicates().sort_values().tolist()

# Explain exactly what this target set contains.
if TARGET_SET_NAME in ['bottom24_plus_guard', 'bottom24_lowrecall_plus_guard', 'all_bad_plus_guard']:
    bottom_ids = set(target_df.loc[target_df['bottom24_edge_rescue'].eq(1), 'sample_id'].astype(str))
    lowrecall_ids = set(target_df.loc[target_df['low_recall_rescue'].eq(1), 'sample_id'].astype(str))
    motion_ids = set(target_df.loc[target_df['motion_gap_conservative'].eq(1), 'sample_id'].astype(str))
    branch_ids = set(target_df.loc[target_df['branch_div_conservative'].eq(1), 'sample_id'].astype(str))
    guard_ids = set(target_df.loc[target_df['guard_high_score'].eq(1), 'sample_id'].astype(str))
    bad_ids = bottom_ids | lowrecall_ids | motion_ids | branch_ids
    union_ids = set(sample_ids)
    print('TARGET_SET_NAME:', TARGET_SET_NAME)
    print('target composition source: reports/targeted_validation_sets.csv')
    print('bottom24 bad count:', len(bottom_ids))
    print('low-recall rescue count:', len(lowrecall_ids))
    print('motion/gap-risk count:', len(motion_ids))
    print('branch/div-risk count:', len(branch_ids))
    print('ALL BAD union count:', len(bad_ids))
    print('high-score guard count:', len(guard_ids))
    print('bad ? guard overlap:', len(bad_ids & guard_ids), sorted(bad_ids & guard_ids)[:10])
    print('guards included:', sorted(guard_ids & union_ids))
    print('union target samples:', len(sample_ids))
else:
    print('TARGET_SET_NAME:', TARGET_SET_NAME)
    print('target samples:', len(sample_ids))
print(sample_ids[:20], '...' if len(sample_ids) > 20 else '')
print('checkpoint geff total:', len(list(CHECKPOINT_DIR.glob('*.geff'))))

# Checkpoint structure preflight for selected samples. This prevents late build failures.
required_rel_paths = [
    'nodes/ids', 'nodes/props/t/values', 'nodes/props/z/values',
    'nodes/props/y/values', 'nodes/props/x/values', 'edges/ids',
]
missing_checkpoint = []
bad_structure = []
for sample_id in sample_ids:
    geff = CHECKPOINT_DIR / f'{sample_id}.geff'
    if not geff.exists():
        missing_checkpoint.append(sample_id)
        continue
    missing_parts = [rel for rel in required_rel_paths if not (geff / rel).exists()]
    if missing_parts:
        bad_structure.append((sample_id, missing_parts))
print('missing checkpoint:', missing_checkpoint)
print('bad checkpoint structure:', bad_structure[:10], 'count=', len(bad_structure))
assert not missing_checkpoint, 'Some selected samples are missing checkpoint predictions.'
assert not bad_structure, 'Some selected checkpoint GEFFs are structurally incomplete.'

# Recreate local pseudo-test folder with symlinks to selected train zarr directories.
for item in LOCAL_TEST_DIR.glob('*'):
    if item.is_symlink() or item.is_file():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

for sample_id in sample_ids:
    src = BASE_DIR / 'train' / f'{sample_id}.zarr'
    dst = LOCAL_TEST_DIR / f'{sample_id}.zarr'
    if not src.exists():
        raise FileNotFoundError(src)
    try:
        os.symlink(src, dst, target_is_directory=True)
    except Exception:
        shutil.copytree(src, dst)

print('pseudo-test zarr count:', len(list(LOCAL_TEST_DIR.glob('*.zarr'))))

# Base environment used by the original public notebook code.
os.environ['BIOHUB_TEST_DIR'] = str(LOCAL_TEST_DIR)
os.environ['BIOHUB_MODEL_ARTIFACTS'] = str(SUPPORT_DIR)
os.environ['BIOHUB_ARTIFACTS'] = str(SUPPORT_DIR)
os.environ['BIOHUB_PRIMARY_ARTIFACT_MANIFEST'] = str(SUPPORT_DIR / 'ARTIFACT_MANIFEST.json')
os.environ['BIOHUB_FULL_FRAME_MANIFEST_DEFAULT'] = str(CENTER_DIR / 'ARTIFACT_MANIFEST.json')
os.environ['BIOHUB_FULL_FRAME_CHECKPOINT_DEFAULT'] = str(CENTER_DIR / 'weights/full_frame_center/checkpoint_last.pt')
os.environ['BIOHUB_ALLOW_PIP_INSTALL'] = '0'
os.environ['BIOHUB_RUN_VISUAL_EDA'] = '0'
os.environ['BIOHUB_RUN_OUTPUT_DIAGNOSTICS'] = '0'

os.chdir(WORK_DIR)
print('cwd:', Path.cwd())


def set_active_variant(variant_name: str):
    if variant_name not in TUNING_VARIANTS:
        raise KeyError(f'Unknown variant: {variant_name}')
    for key in VARIANT_ENV_KEYS:
        os.environ.pop(key, None)
    for key, value in TUNING_VARIANTS[variant_name].items():
        os.environ[key] = str(value)
    global SELECTED_VARIANT, OUTPUT_PREFIX, CLEAR_OLD_PREDICTIONS, WRITE_STABLE_ALIAS
    SELECTED_VARIANT = variant_name
    OUTPUT_PREFIX = f'deepcenter_target_{TARGET_SET_NAME}_{variant_name}'
    CLEAR_OLD_PREDICTIONS = variant_name == VARIANTS_TO_RUN[0]
    WRITE_STABLE_ALIAS = False
    print('\n' + '=' * 90)
    print('ACTIVE VARIANT:', SELECTED_VARIANT)
    print('OUTPUT_PREFIX:', OUTPUT_PREFIX)
    print('variant env:', json.dumps(TUNING_VARIANTS[variant_name], indent=2, sort_keys=True))

## Embedded Public Pipeline Code

The next cell stores the already-working public DeepCenter config/build/copy/score cells as strings. This lets the notebook run variants in a loop while keeping the original implementation intact.

In [ ]:
CONFIG_CODE = 'from __future__ import annotations\n\nimport csv\nimport importlib.util\nimport json\nimport math\nimport os\nimport shutil\nimport subprocess\nimport tempfile\nimport zipfile\nimport sys\nimport time\nfrom pathlib import Path\n\nimport pandas as pd\n\nCOMPETITION = "biohub-cell-tracking-during-development"\nCOMP_DIR_CANDIDATES = [\n    Path(f"/kaggle/input/competitions/{COMPETITION}"),\n    Path(f"/kaggle/input/{COMPETITION}"),\n]\nCOMP_DIR = next((path for path in COMP_DIR_CANDIDATES if path.exists()), COMP_DIR_CANDIDATES[0])\n_test_dir_override = os.environ.get("BIOHUB_TEST_DIR", "").strip()\nTEST_DIR = Path(_test_dir_override) if _test_dir_override else COMP_DIR / "test"\n\nWORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")\nREPO_DIR = WORKING_DIR / "tracking_repo"\nSUBMISSION_PATH = WORKING_DIR / "submission.csv"\nRUN_STATS_PATH = WORKING_DIR / "run_stats.csv"\n\nMETHOD = "unet_transformer"\nWEIGHTS_RELATIVE = f"weights/{METHOD}/split_0/edge_predictor_best.pth"\nEXPERIMENT_TAG = "selected_30_deepcenter_500_last_sparse_gate_min7"\nTARGET_ARTIFACT_SLUG = os.environ.get("BIOHUB_TARGET_ARTIFACT_SLUG", "biohub-tracking-support-pack-50ep-v1")\nPRIMARY_ARTIFACT_MANIFEST = Path(os.environ.get(\n    "BIOHUB_PRIMARY_ARTIFACT_MANIFEST",\n    "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/ARTIFACT_MANIFEST.json",\n))\nALLOW_ARTIFACT_FALLBACK = os.environ.get("BIOHUB_ALLOW_ARTIFACT_FALLBACK", "0") != "0"\n\nDET_THRESHOLD = float(os.environ.get("BIOHUB_DET_THRESHOLD", "0.99"))\nUNET_BATCH_SIZE = int(os.environ.get("BIOHUB_UNET_BATCH_SIZE", "4"))\nUSE_ILP = os.environ.get("BIOHUB_USE_ILP", "1") != "0"\nILP_EDGE_WEIGHT = float(os.environ.get("BIOHUB_ILP_EDGE_WEIGHT", "-1.0"))\nILP_APPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_APPEARANCE_WEIGHT", "0.1"))\nILP_DISAPPEARANCE_WEIGHT = float(os.environ.get("BIOHUB_ILP_DISAPPEARANCE_WEIGHT", "0.1"))\nILP_DIVISION_WEIGHT = float(os.environ.get("BIOHUB_ILP_DIVISION_WEIGHT", "1.0"))\n\n# Empty for a real submission. Useful for local smoke tests, e.g. BIOHUB_SLICE=:1.\nSLICE = os.environ.get("BIOHUB_SLICE", "").strip()\n\n# If dependencies are not already installed and no offline wheels are attached,\n# this controls whether the notebook attempts PyPI installation.\nALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"\nRUN_OUTPUT_DIAGNOSTICS = os.environ.get("BIOHUB_RUN_OUTPUT_DIAGNOSTICS", "1") != "0"\nRUN_VISUAL_EDA = os.environ.get("BIOHUB_RUN_VISUAL_EDA", "1") != "0"\n\n# Optional detector fusion. This stays off automatically when the support\n# artifact has no weights/full_frame_center/best.pt checkpoint.\nUSE_FULL_FRAME_CENTER_FUSION = os.environ.get("BIOHUB_USE_FULL_FRAME_CENTER_FUSION", "1") != "0"\nREQUIRE_FULL_FRAME_CENTER = os.environ.get("BIOHUB_REQUIRE_FULL_FRAME_CENTER", "1") != "0"\nFULL_FRAME_CENTER_RELATIVE = os.environ.get("BIOHUB_FULL_FRAME_CENTER_RELATIVE", "weights/full_frame_center/best.pt")\nFULL_FRAME_MANIFEST_DEFAULT = os.environ.get(\n    "BIOHUB_FULL_FRAME_MANIFEST_DEFAULT",\n    "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/ARTIFACT_MANIFEST.json",\n)\nFULL_FRAME_CHECKPOINT_DEFAULT = os.environ.get(\n    "BIOHUB_FULL_FRAME_CHECKPOINT_DEFAULT",\n    "/kaggle/input/biohub-deepcenter-unet3d-center-prior-v1/weights/full_frame_center/checkpoint_last.pt",\n)\nFULL_FRAME_PEAK_THRESHOLD = float(os.environ.get("BIOHUB_FULL_FRAME_PEAK_THRESHOLD", "0.15"))\nFULL_FRAME_MIN_PEAK_DISTANCE = int(os.environ.get("BIOHUB_FULL_FRAME_MIN_PEAK_DISTANCE", "1"))\nFULL_FRAME_NMS_RADIUS_UM = float(os.environ.get("BIOHUB_FULL_FRAME_NMS_RADIUS_UM", "4.4"))\nFULL_FRAME_EXISTING_GATE_UM = float(os.environ.get("BIOHUB_FULL_FRAME_EXISTING_GATE_UM", "5.0"))\nFULL_FRAME_MAX_ADDED_FRAC = float(os.environ.get("BIOHUB_FULL_FRAME_MAX_ADDED_FRAC", "0.008"))\nFULL_FRAME_MAX_ADDED_ABS = int(os.environ.get("BIOHUB_FULL_FRAME_MAX_ADDED_ABS", "25"))\nFULL_FRAME_REFINE_CENTROIDS = os.environ.get("BIOHUB_FULL_FRAME_REFINE_CENTROIDS", "1") != "0"\nFULL_FRAME_USE_GATE_CALIBRATION = os.environ.get("BIOHUB_FULL_FRAME_USE_GATE_CALIBRATION", "0") != "0"\nFULL_FRAME_GATE_TARGET_PRECISION = float(os.environ.get("BIOHUB_FULL_FRAME_GATE_TARGET_PRECISION", "0.07"))\nFULL_FRAME_GATE_MAX_PRED_PER_FRAME = float(os.environ.get("BIOHUB_FULL_FRAME_GATE_MAX_PRED_PER_FRAME", "0"))\n\n# Output-level graph post-processing.\nOUTPUT_EDGE_MAX_UM = float(os.environ.get("BIOHUB_OUTPUT_EDGE_MAX_UM", "14.0"))\nOUTPUT_ENFORCE_NEXT_FRAME = os.environ.get("BIOHUB_OUTPUT_ENFORCE_NEXT_FRAME", "1") != "0"\nOUTPUT_SINGLE_PARENT_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_PARENT_REPAIR", "1") != "0"\nOUTPUT_SINGLE_CHILD_REPAIR = os.environ.get("BIOHUB_OUTPUT_SINGLE_CHILD_REPAIR", "0") != "0"\nOUTPUT_PRUNE_ISOLATED = os.environ.get("BIOHUB_OUTPUT_PRUNE_ISOLATED", "1") != "0"\nOUTPUT_MOTION_RELINK = os.environ.get("BIOHUB_OUTPUT_MOTION_RELINK", "1") != "0"\nMOTION_RELINK_TIGHT_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_TIGHT_UM", "6.0"))\nMOTION_RELINK_RELAXED_UM = float(os.environ.get("BIOHUB_MOTION_RELINK_RELAXED_UM", "10.0"))\nMOTION_RELINK_VELOCITY_WEIGHT = float(os.environ.get("BIOHUB_MOTION_RELINK_VELOCITY_WEIGHT", "0.5"))\nMOTION_RELINK_LEARNED_BONUS = float(os.environ.get("BIOHUB_MOTION_RELINK_LEARNED_BONUS", "0.75"))\nMOTION_RELINK_MAX_FRAME_NODES = int(os.environ.get("BIOHUB_MOTION_RELINK_MAX_FRAME_NODES", "2600"))\n\nOUTPUT_DIVISION_GEOMETRY_FILTER = os.environ.get("BIOHUB_OUTPUT_DIVISION_GEOMETRY_FILTER", "0") != "0"\nDIV_PARENT_MAX_UM = float(os.environ.get("BIOHUB_DIV_PARENT_MAX_UM", "10.5"))\nDIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_DIV_SISTER_MAX_UM", "8.0"))\nDIV_DROP_TO_SINGLE_IF_BAD = os.environ.get("BIOHUB_DIV_DROP_TO_SINGLE_IF_BAD", "1") != "0"\nOUTPUT_GAP_CLOSE = os.environ.get("BIOHUB_OUTPUT_GAP_CLOSE", "1") != "0"\nGAP_CLOSE_MAX_GAP = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_GAP", "1"))\nGAP_CLOSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_UM", "6.0"))\nGAP_CLOSE_REUSE_EXISTING = os.environ.get("BIOHUB_GAP_CLOSE_REUSE_EXISTING", "1") != "0"\nGAP_CLOSE_REUSE_UM = float(os.environ.get("BIOHUB_GAP_CLOSE_REUSE_UM", "3.2"))\nGAP_CLOSE_MAX_ADDED_FRAC = float(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_FRAC", "0.05"))\nGAP_CLOSE_MAX_ADDED_ABS = int(os.environ.get("BIOHUB_GAP_CLOSE_MAX_ADDED_ABS", "2000"))\nGAP_REFINE_SYNTHETIC = os.environ.get("BIOHUB_GAP_REFINE_SYNTHETIC", "1") != "0"\nGAP_REFINE_WIN_Z = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_Z", "1"))\nGAP_REFINE_WIN_YX = int(os.environ.get("BIOHUB_GAP_REFINE_WIN_YX", "3"))\nGAP_REFINE_MAX_SHIFT_UM = float(os.environ.get("BIOHUB_GAP_REFINE_MAX_SHIFT_UM", "3.2"))\n\nOUTPUT_FILTER_SHORT_TRACKS = os.environ.get("BIOHUB_OUTPUT_FILTER_SHORT_TRACKS", "1") != "0"\nOUTPUT_MIN_TRACK_LEN = int(os.environ.get("BIOHUB_OUTPUT_MIN_TRACK_LEN", "7"))\nOUTPUT_KEEP_DIVISION_COMPONENTS = os.environ.get("BIOHUB_OUTPUT_KEEP_DIVISION_COMPONENTS", "1") != "0"\n\nOUTPUT_LINEFIT_SMOOTH = os.environ.get("BIOHUB_OUTPUT_LINEFIT_SMOOTH", "1") != "0"\nOUTPUT_LINEFIT_WEIGHT = float(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WEIGHT", "0.8"))\nOUTPUT_LINEFIT_WINDOW = int(os.environ.get("BIOHUB_OUTPUT_LINEFIT_WINDOW", "2"))\n\nOUTPUT_GAP2_RECOVERY = os.environ.get("BIOHUB_OUTPUT_GAP2_RECOVERY", "0") != "0"\nGAP2_MAX_TOTAL_UM = float(os.environ.get("BIOHUB_GAP2_MAX_TOTAL_UM", "10.2"))\nGAP2_MAX_STEP_UM = float(os.environ.get("BIOHUB_GAP2_MAX_STEP_UM", "4.4"))\nGAP2_MAX_LINKS_FRAC = float(os.environ.get("BIOHUB_GAP2_MAX_LINKS_FRAC", "0.0045"))\nGAP2_MAX_LINKS_ABS = int(os.environ.get("BIOHUB_GAP2_MAX_LINKS_ABS", "180"))\nGAP2_REQUIRE_CONTEXT = os.environ.get("BIOHUB_GAP2_REQUIRE_CONTEXT", "1") != "0"\nGAP2_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_GAP2_FRAME_FRAC_CAP", "0.006"))\n\nOUTPUT_SAFE_DIVISIONS = os.environ.get("BIOHUB_OUTPUT_SAFE_DIVISIONS", "1") != "0"\nSAFE_DIV_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_MAX_UM", "4.7"))\nSAFE_DIV_SISTER_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_SISTER_MAX_UM", "7.2"))\nSAFE_DIV_EXISTING_CHILD_MAX_UM = float(os.environ.get("BIOHUB_SAFE_DIV_EXISTING_CHILD_MAX_UM", "7.8"))\nSAFE_DIV_FRAME_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_FRAME_FRAC_CAP", "0.008"))\nSAFE_DIV_GLOBAL_FRAC_CAP = float(os.environ.get("BIOHUB_SAFE_DIV_GLOBAL_FRAC_CAP", "0.004"))\n\nCONFIG_DISPLAY = {\n    "experiment_tag": EXPERIMENT_TAG,\n    "method": METHOD,\n    "weights": WEIGHTS_RELATIVE,\n    "target_artifact_slug": TARGET_ARTIFACT_SLUG,\n    "primary_artifact_manifest": str(PRIMARY_ARTIFACT_MANIFEST),\n    "allow_artifact_fallback": ALLOW_ARTIFACT_FALLBACK,\n    "det_threshold": DET_THRESHOLD,\n    "unet_batch_size": UNET_BATCH_SIZE,\n    "use_ilp": USE_ILP,\n    "ilp_edge_weight": ILP_EDGE_WEIGHT,\n    "ilp_appearance_weight": ILP_APPEARANCE_WEIGHT,\n    "ilp_disappearance_weight": ILP_DISAPPEARANCE_WEIGHT,\n    "ilp_division_weight": ILP_DIVISION_WEIGHT,\n    "slice": SLICE,\n    "allow_pip_install": ALLOW_PIP_INSTALL,\n    "run_visual_eda": RUN_VISUAL_EDA,\n    "use_full_frame_center_fusion": USE_FULL_FRAME_CENTER_FUSION,\n    "require_full_frame_center": REQUIRE_FULL_FRAME_CENTER,\n    "full_frame_center_relative": FULL_FRAME_CENTER_RELATIVE,\n    "full_frame_manifest_default": FULL_FRAME_MANIFEST_DEFAULT,\n    "full_frame_checkpoint_default": FULL_FRAME_CHECKPOINT_DEFAULT,\n    "full_frame_peak_threshold": FULL_FRAME_PEAK_THRESHOLD,\n    "full_frame_min_peak_distance": FULL_FRAME_MIN_PEAK_DISTANCE,\n    "full_frame_nms_radius_um": FULL_FRAME_NMS_RADIUS_UM,\n    "full_frame_existing_gate_um": FULL_FRAME_EXISTING_GATE_UM,\n    "full_frame_max_added_frac": FULL_FRAME_MAX_ADDED_FRAC,\n    "full_frame_max_added_abs": FULL_FRAME_MAX_ADDED_ABS,\n    "full_frame_refine_centroids": FULL_FRAME_REFINE_CENTROIDS,\n    "full_frame_use_gate_calibration": FULL_FRAME_USE_GATE_CALIBRATION,\n    "full_frame_gate_target_precision": FULL_FRAME_GATE_TARGET_PRECISION,\n    "full_frame_gate_max_pred_per_frame": FULL_FRAME_GATE_MAX_PRED_PER_FRAME,\n    "output_edge_max_um": OUTPUT_EDGE_MAX_UM,\n    "output_enforce_next_frame": OUTPUT_ENFORCE_NEXT_FRAME,\n    "output_single_parent_repair": OUTPUT_SINGLE_PARENT_REPAIR,\n    "output_single_child_repair": OUTPUT_SINGLE_CHILD_REPAIR,\n    "output_prune_isolated": OUTPUT_PRUNE_ISOLATED,\n    "output_motion_relink": OUTPUT_MOTION_RELINK,\n    "motion_relink_tight_um": MOTION_RELINK_TIGHT_UM,\n    "motion_relink_relaxed_um": MOTION_RELINK_RELAXED_UM,\n    "motion_relink_velocity_weight": MOTION_RELINK_VELOCITY_WEIGHT,\n    "motion_relink_learned_bonus": MOTION_RELINK_LEARNED_BONUS,\n    "motion_relink_max_frame_nodes": MOTION_RELINK_MAX_FRAME_NODES,\n    "output_division_geometry_filter": OUTPUT_DIVISION_GEOMETRY_FILTER,\n    "div_parent_max_um": DIV_PARENT_MAX_UM,\n    "div_sister_max_um": DIV_SISTER_MAX_UM,\n    "div_drop_to_single_if_bad": DIV_DROP_TO_SINGLE_IF_BAD,\n    "output_gap_close": OUTPUT_GAP_CLOSE,\n    "gap_close_max_gap": GAP_CLOSE_MAX_GAP,\n    "gap_close_effective_max_gap": min(GAP_CLOSE_MAX_GAP, 1),\n    "gap_close_um": GAP_CLOSE_UM,\n    "gap_close_reuse_existing": GAP_CLOSE_REUSE_EXISTING,\n    "gap_close_reuse_um": GAP_CLOSE_REUSE_UM,\n    "gap_close_max_added_frac": GAP_CLOSE_MAX_ADDED_FRAC,\n    "gap_close_max_added_abs": GAP_CLOSE_MAX_ADDED_ABS,\n    "gap_refine_synthetic": GAP_REFINE_SYNTHETIC,\n    "gap_refine_win_z": GAP_REFINE_WIN_Z,\n    "gap_refine_win_yx": GAP_REFINE_WIN_YX,\n    "gap_refine_max_shift_um": GAP_REFINE_MAX_SHIFT_UM,\n    "output_filter_short_tracks": OUTPUT_FILTER_SHORT_TRACKS,\n    "output_min_track_len": OUTPUT_MIN_TRACK_LEN,\n    "output_keep_division_components": OUTPUT_KEEP_DIVISION_COMPONENTS,\n    "output_linefit_smooth": OUTPUT_LINEFIT_SMOOTH,\n    "output_linefit_weight": OUTPUT_LINEFIT_WEIGHT,\n    "output_linefit_window": OUTPUT_LINEFIT_WINDOW,\n    "output_gap2_recovery": OUTPUT_GAP2_RECOVERY,\n    "gap2_max_total_um": GAP2_MAX_TOTAL_UM,\n    "gap2_max_step_um": GAP2_MAX_STEP_UM,\n    "gap2_max_links_frac": GAP2_MAX_LINKS_FRAC,\n    "gap2_max_links_abs": GAP2_MAX_LINKS_ABS,\n    "gap2_require_context": GAP2_REQUIRE_CONTEXT,\n    "gap2_frame_frac_cap": GAP2_FRAME_FRAC_CAP,\n    "output_safe_divisions": OUTPUT_SAFE_DIVISIONS,\n    "safe_div_max_um": SAFE_DIV_MAX_UM,\n    "safe_div_sister_max_um": SAFE_DIV_SISTER_MAX_UM,\n    "safe_div_existing_child_max_um": SAFE_DIV_EXISTING_CHILD_MAX_UM,\n    "safe_div_frame_frac_cap": SAFE_DIV_FRAME_FRAC_CAP,\n    "safe_div_global_frac_cap": SAFE_DIV_GLOBAL_FRAC_CAP,\n}\n\nprint("Biohub learned UNet + node-transformer + ILP submission")\nprint("COMP_DIR:", COMP_DIR, "exists:", COMP_DIR.exists())\nprint("TEST_DIR:", TEST_DIR, "exists:", TEST_DIR.exists())\nprint(json.dumps(CONFIG_DISPLAY, indent=2, sort_keys=True))'
DEPENDENCY_CODE = 'import re\n\nos.environ.setdefault("POLARS_PREFER_PKG", "32")\n\nPACKAGE_SPECS = {\n    "tracksdata": ("tracksdata", "tracksdata"),\n    "zarr": ("zarr", "zarr>=3.0.10,<4"),\n    "pyscipopt": ("pyscipopt", "pyscipopt"),\n    "geff": ("geff", "geff>=1.1.3.1.1"),\n    "geff_spec": ("geff_spec", "geff-spec<1.2"),\n    "ilpy": ("ilpy", "ilpy>=0.5.1"),\n    "polars": ("polars", "polars>=1.36"),\n    "blosc2": ("blosc2", "blosc2"),\n    "dask": ("dask", "dask"),\n    "imagecodecs": ("imagecodecs", "imagecodecs"),\n    "skimage": ("skimage", "scikit-image>=0.24"),\n    "pyarrow": ("pyarrow", "pyarrow"),\n    "rustworkx": ("rustworkx", "rustworkx>=0.17.1"),\n    "sqlalchemy": ("sqlalchemy", "sqlalchemy>=2"),\n    "numcodecs": ("numcodecs", "numcodecs>=0.13,<0.16"),\n    "donfig": ("donfig", "donfig>=0.8"),\n    "google_crc32c": ("google_crc32c", "google-crc32c>=1.5"),\n    "bidict": ("bidict", "bidict>=0.23.1"),\n    "psygnal": ("psygnal", "psygnal>=0.14"),\n    "rich": ("rich", "rich"),\n    "networkx": ("networkx", "networkx>=3.2.1"),\n    "pydantic": ("pydantic", "pydantic>=2.11"),\n    "pydantic_core": ("pydantic_core", "pydantic-core"),\n    "annotated_types": ("annotated_types", "annotated-types"),\n    "typing_extensions": ("typing_extensions", "typing-extensions>=4.13"),\n    "typing_inspection": ("typing_inspection", "typing-inspection"),\n    "markdown_it": ("markdown_it", "markdown-it-py"),\n    "pygments": ("pygments", "pygments"),\n    "click": ("click", "click"),\n    "cloudpickle": ("cloudpickle", "cloudpickle"),\n    "fsspec": ("fsspec", "fsspec"),\n    "partd": ("partd", "partd"),\n    "locket": ("locket", "locket"),\n    "toolz": ("toolz", "toolz"),\n    "yaml": ("yaml", "pyyaml"),\n    "ndindex": ("ndindex", "ndindex"),\n    "msgpack": ("msgpack", "msgpack"),\n    "numexpr": ("numexpr", "numexpr"),\n    "deprecated": ("deprecated", "deprecated"),\n    "wrapt": ("wrapt", "wrapt"),\n    "imageio": ("imageio", "imageio"),\n    "PIL": ("PIL", "pillow"),\n    "tifffile": ("tifffile", "tifffile"),\n    "lazy_loader": ("lazy_loader", "lazy-loader"),\n    "tqdm": ("tqdm", "tqdm"),\n}\nEXTRA_SPECS_BY_NAME = {\n    "tracksdata": ["bidict>=0.23.1", "psygnal>=0.14", "rich"],\n    "zarr": ["donfig>=0.8", "google-crc32c>=1.5", "numcodecs>=0.13,<0.16"],\n    "geff": ["geff-spec<1.2", "networkx>=3.2.1", "pydantic>=2.11", "numcodecs>=0.13,<0.16"],\n    "geff_spec": ["pydantic>=2.11", "annotated-types", "pydantic-core", "typing-inspection"],\n    "polars": ["polars-runtime-32"],\n    "dask": ["click", "cloudpickle", "fsspec", "partd", "pyyaml", "toolz"],\n    "partd": ["locket"],\n    "blosc2": ["ndindex", "msgpack", "numexpr"],\n    "numcodecs": ["deprecated", "msgpack", "wrapt"],\n    "rich": ["markdown-it-py", "pygments"],\n    "pydantic": ["annotated-types", "pydantic-core", "typing-extensions>=4.13", "typing-inspection"],\n    "skimage": ["imageio", "pillow", "tifffile", "lazy-loader", "networkx"],\n}\nPIP_DEPENDENCIES = [spec for _, spec in PACKAGE_SPECS.values()]\nREQUIRED_MODULES = {name: module for name, (module, _) in PACKAGE_SPECS.items() if module}\nFALLBACK_ARTIFACT_SLUGS = ["biohub-tracking-support-pack-v1"]\n\n# The safe path for offline reruns is to use attached wheels.\n# Set BIOHUB_ALLOW_PIP_INSTALL=1 only for an interactive internet-enabled run.\nALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"\n\n\ndef module_missing(module_name: str) -> bool:\n    return importlib.util.find_spec(module_name) is None\n\n\ndef has_model_artifact(path: Path) -> bool:\n    has_repo_dir = (path / "repo").exists()\n    has_weights_dir = (path / "weights" / METHOD / "split_0" / "edge_predictor_best.pth").exists()\n    has_repo_zip = (path / "repo.zip").exists()\n    has_weights_zip = (path / "weights.zip").exists()\n    return (has_repo_dir and has_weights_dir) or (has_repo_zip and has_weights_zip)\n\n\ndef artifact_manifest(path: Path) -> dict:\n    manifest = path / "ARTIFACT_MANIFEST.json"\n    if not manifest.exists():\n        return {}\n    try:\n        return json.loads(manifest.read_text())\n    except Exception:\n        return {}\n\n\ndef artifact_matches_target(path: Path) -> bool:\n    if ALLOW_ARTIFACT_FALLBACK:\n        return True\n    manifest = artifact_manifest(path)\n    artifact_name = str(manifest.get("artifact_name", ""))\n    path_text = str(path)\n    return TARGET_ARTIFACT_SLUG in {artifact_name, path.name} or TARGET_ARTIFACT_SLUG in path_text\n\n\ndef candidate_roots_for_slug(slug: str) -> list[Path]:\n    return [\n        Path(f"/kaggle/input/datasets/pilkwang/{slug}"),\n        Path(f"/kaggle/input/{slug}"),\n        Path(f"/kaggle/input/{slug}/{slug}"),\n        Path(f"PublicNotebook/{slug}"),\n    ]\n\n\ndef find_artifacts_root() -> Path:\n    candidates: list[Path] = []\n    for env_name in ["BIOHUB_MODEL_ARTIFACTS", "BIOHUB_ARTIFACTS"]:\n        explicit = os.environ.get(env_name, "").strip()\n        if explicit:\n            candidates.append(Path(explicit))\n\n    candidates.append(PRIMARY_ARTIFACT_MANIFEST.parent)\n    candidates.extend(candidate_roots_for_slug(TARGET_ARTIFACT_SLUG))\n\n    if ALLOW_ARTIFACT_FALLBACK:\n        for slug in FALLBACK_ARTIFACT_SLUGS:\n            candidates.extend(candidate_roots_for_slug(slug))\n\n    input_root = Path("/kaggle/input")\n    if input_root.exists():\n        for child in input_root.iterdir():\n            if not child.is_dir():\n                continue\n            child_text = str(child)\n            if TARGET_ARTIFACT_SLUG in child_text or ALLOW_ARTIFACT_FALLBACK:\n                candidates.append(child)\n                candidates.append(child / child.name)\n                for grandchild in child.iterdir():\n                    if grandchild.is_dir():\n                        candidates.append(grandchild)\n\n    seen: set[Path] = set()\n    for candidate in candidates:\n        candidate = candidate.expanduser()\n        if candidate in seen:\n            continue\n        seen.add(candidate)\n        if has_model_artifact(candidate) and artifact_matches_target(candidate):\n            return candidate\n    checked = "\\n".join(str(path) for path in candidates[:80])\n    raise FileNotFoundError(\n        "Could not find the required model artifact. "\n        f"Expected slug: {TARGET_ARTIFACT_SLUG}\\n"\n        "Attach the newly uploaded support dataset, or set BIOHUB_MODEL_ARTIFACTS.\\n"\n        "To debug with an older artifact, set BIOHUB_ALLOW_ARTIFACT_FALLBACK=1.\\n"\n        "Checked:\\n" + checked\n    )\n\n\ndef _has_package_file(path: Path) -> bool:\n    if not path.exists() or not path.is_dir():\n        return False\n    patterns = ("*.whl", "*.tar.gz", "*.zip")\n    return any(any(path.glob(pattern)) for pattern in patterns)\n\n\ndef find_offline_package_dirs(artifacts: Path) -> list[Path]:\n    candidates: list[Path] = [\n        artifacts / "wheels",\n        artifacts,\n        Path("/kaggle/working"),\n        Path("/kaggle/working/wheels"),\n    ]\n    input_root = Path("/kaggle/input")\n    if input_root.exists():\n        for child in input_root.iterdir():\n            if child.is_dir():\n                candidates.extend([child / "wheels", child])\n                for grandchild in child.iterdir():\n                    if grandchild.is_dir():\n                        candidates.extend([grandchild / "wheels", grandchild])\n\n    out: list[Path] = []\n    seen: set[Path] = set()\n    for candidate in candidates:\n        candidate = candidate.expanduser()\n        if candidate in seen:\n            continue\n        seen.add(candidate)\n        if _has_package_file(candidate):\n            out.append(candidate)\n    return out\n\n\ndef purge_imported_modules(package_names: list[str]) -> None:\n    roots = {"tracksdata"}\n    for name in package_names:\n        if name in PACKAGE_SPECS:\n            module = PACKAGE_SPECS[name][0]\n            roots.add(module.split(".")[0])\n        if name == "polars":\n            roots.add("polars")\n    for root in roots:\n        for module_name in list(sys.modules):\n            if module_name == root or module_name.startswith(root + "."):\n                sys.modules.pop(module_name, None)\n\n\ndef polars_runtime_ready() -> bool:\n    try:\n        import polars as _pl\n        from polars._plr import PySeries as _PySeries\n\n        _ = _PySeries\n        return hasattr(_pl, "Float16") and _pl.Series([-999999.0], dtype=_pl.Float64).dtype == _pl.Float64\n    except Exception:\n        return False\n\n\ndef packages_requiring_refresh() -> list[str]:\n    refresh: list[str] = []\n    if not module_missing("polars") and not polars_runtime_ready():\n        refresh.append("polars")\n\n    if not module_missing("zarr"):\n        try:\n            import zarr as _zarr\n            version_text = str(getattr(_zarr, "__version__", "0"))\n            major = int(version_text.split(".", 1)[0])\n            if major < 3:\n                refresh.append("zarr")\n        except Exception:\n            refresh.append("zarr")\n    return refresh\n\n\ndef dependency_specs_for(missing: list[str]) -> list[str]:\n    specs: list[str] = []\n    seen: set[str] = set()\n\n    def add(spec: str) -> None:\n        key = spec.lower()\n        if key not in seen:\n            seen.add(key)\n            specs.append(spec)\n\n    for name in missing:\n        if name in PACKAGE_SPECS:\n            add(PACKAGE_SPECS[name][1])\n        for spec in EXTRA_SPECS_BY_NAME.get(name, []):\n            add(spec)\n    return specs\n\n\ndef import_failures() -> dict[str, str]:\n    failures: dict[str, str] = {}\n    for name, module_name in REQUIRED_MODULES.items():\n        try:\n            importlib.import_module(module_name)\n        except Exception as exc:\n            failures[name] = f"{type(exc).__name__}: {exc}"\n    return failures\n\n\ndef missing_names_from_failures(failures: dict[str, str]) -> list[str]:\n    names: list[str] = []\n    module_to_name = {module: name for name, module in REQUIRED_MODULES.items()}\n    for message in failures.values():\n        match = re.search(r"No module named [\'\\"]([^\'\\"]+)[\'\\"]", message)\n        if match:\n            module = match.group(1).split(".")[0]\n        else:\n            match = re.search(r"module [\'\\"]([^\'\\"]+)[\'\\"] has no attribute", message)\n            if not match:\n                continue\n            module = match.group(1).split(".")[0]\n        name = module_to_name.get(module)\n        if name and name not in names:\n            names.append(name)\n    return names\n\n\ndef install_missing_dependencies(missing: list[str], artifacts: Path) -> None:\n    specs = dependency_specs_for(missing)\n    force_reinstall = bool({"polars", "zarr"} & set(missing))\n    if not specs:\n        return\n\n    package_dirs = find_offline_package_dirs(artifacts)\n    if package_dirs:\n        offline_cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"]\n        if force_reinstall:\n            offline_cmd.append("--force-reinstall")\n        for package_dir in package_dirs:\n            offline_cmd.extend(["--find-links", str(package_dir)])\n        offline_cmd.extend(specs)\n        print("Installing missing packages from offline package dirs:", missing)\n        print("Dependency resolver is disabled with --no-deps to avoid replacing Kaggle numpy/scipy in a live kernel.")\n        print("Offline package dirs:", [str(path) for path in package_dirs])\n        result = subprocess.run(offline_cmd, text=True, capture_output=True)\n        if result.returncode == 0:\n            purge_imported_modules(missing)\n            print("Offline dependency install succeeded.")\n            return\n        print("Offline dependency install failed. Last pip output:")\n        print((result.stdout or "")[-2000:])\n        print((result.stderr or "")[-2000:])\n\n    if ALLOW_PIP_INSTALL:\n        online_cmd = [sys.executable, "-m", "pip", "install", "--no-deps"]\n        if force_reinstall:\n            online_cmd.append("--force-reinstall")\n        online_cmd.extend(specs)\n        print("Installing missing packages from PyPI:", missing)\n        result = subprocess.run(online_cmd, text=True, capture_output=True)\n        if result.returncode == 0:\n            purge_imported_modules(missing)\n            print("PyPI dependency install succeeded.")\n            return\n        print("PyPI dependency install failed. Last pip output:")\n        print((result.stdout or "")[-2000:])\n        print((result.stderr or "")[-2000:])\n\n    command = "pip install tracksdata zarr>=3.0.10,<4 pyscipopt geff geff-spec ilpy polars blosc2 dask imagecodecs pyarrow rustworkx sqlalchemy donfig numcodecs"\n    raise ImportError(\n        "Missing required packages or dependency wheels: " + ", ".join(missing) + "\\n"\n        "Attach the support dataset with offline wheels. If supplying Kaggle dependency input instead, use:\\n"\n        + command + "\\n"\n        "Do not quote zarr>=3.0.10,<4 in Kaggle dependency input."\n    )\n\n\ndef ensure_dependencies(artifacts: Path) -> None:\n    for _ in range(5):\n        refresh = packages_requiring_refresh()\n        if refresh:\n            install_missing_dependencies(refresh, artifacts)\n            continue\n\n        missing = [pkg for pkg, module in REQUIRED_MODULES.items() if module_missing(module)]\n        if missing:\n            install_missing_dependencies(missing, artifacts)\n            continue\n\n        failures = import_failures()\n        if not failures:\n            print("Required graph/Zarr/ILP packages import successfully.")\n            return\n\n        missing_from_import = missing_names_from_failures(failures)\n        if missing_from_import:\n            install_missing_dependencies(missing_from_import, artifacts)\n            continue\n\n        raise ImportError(\n            "Required packages are present but failed to import. "\n            "This may indicate a binary dependency mismatch in the live notebook kernel. "\n            "Keep Kaggle dependency input empty and attach the wheels artifact.\\n"\n            + json.dumps(failures, indent=2)\n        )\n\n    failures = import_failures()\n    raise ImportError(\n        "Dependency recovery did not converge after repeated offline installs. "\n        "The attached support artifact may be missing wheels.\\n"\n        + json.dumps(failures, indent=2)\n    )\n\n\ndef remove_path(path: Path) -> None:\n    if path.is_symlink() or path.is_file():\n        path.unlink()\n    elif path.exists():\n        shutil.rmtree(path)\n\n\ndef copy_or_extract_tree(src_dir: Path, src_zip: Path, dst: Path) -> None:\n    remove_path(dst)\n    if src_dir.exists() and src_dir.is_dir():\n        shutil.copytree(src_dir, dst)\n        return\n    if src_zip.exists() and src_zip.is_file():\n        dst.mkdir(parents=True, exist_ok=True)\n        with zipfile.ZipFile(src_zip) as zf:\n            zf.extractall(dst)\n        return\n    raise FileNotFoundError(f"Missing source tree or zip: {src_dir} / {src_zip}")\n\n\ndef link_or_copy_tree(src: Path, dst: Path) -> None:\n    remove_path(dst)\n    try:\n        os.symlink(src, dst, target_is_directory=True)\n    except Exception:\n        shutil.copytree(src, dst)\n\n\ndef materialize_inference_repo(artifacts: Path) -> None:\n    copy_or_extract_tree(artifacts / "repo", artifacts / "repo.zip", REPO_DIR)\n\n    weights_src = artifacts / "weights"\n    weights_zip = artifacts / "weights.zip"\n    weights_dst = REPO_DIR / "weights"\n    if weights_src.exists() and weights_src.is_dir():\n        link_or_copy_tree(weights_src, weights_dst)\n    elif weights_zip.exists() and weights_zip.is_file():\n        remove_path(weights_dst)\n        weights_dst.mkdir(parents=True, exist_ok=True)\n        with zipfile.ZipFile(weights_zip) as zf:\n            zf.extractall(weights_dst)\n    else:\n        raise FileNotFoundError(f"Missing weights tree or zip under {artifacts}")\n\n    required = [\n        REPO_DIR / "scripts" / "predict_unet_transformer.py",\n        REPO_DIR / WEIGHTS_RELATIVE,\n    ]\n    missing = [str(path) for path in required if not path.exists()]\n    if missing:\n        raise FileNotFoundError("Materialized inference repo is incomplete:\\n" + "\\n".join(missing))\n    print("Inference repo:", REPO_DIR)\n    print("Weights:", REPO_DIR / WEIGHTS_RELATIVE)\n\n\nARTIFACTS = find_artifacts_root()\nprint("ARTIFACTS:", ARTIFACTS)\nprint("Has offline wheels:", (ARTIFACTS / "wheels").exists())\nmanifest_info = artifact_manifest(ARTIFACTS)\nif manifest_info:\n    print("Artifact name:", manifest_info.get("artifact_name"))\n    print("Weight sha256:", manifest_info.get("model", {}).get("weight_sha256"))\n    print("Weight path:", manifest_info.get("model", {}).get("weight_path"))\n\nensure_dependencies(ARTIFACTS)\nmaterialize_inference_repo(ARTIFACTS)\n'
RESTORE_CODE = "from pathlib import Path\nimport shutil\nfrom tqdm.auto import tqdm\n\n\ndef list_test_stems() -> list[str]:\n    if not TEST_DIR.exists():\n        raise FileNotFoundError(f'Test directory does not exist: {TEST_DIR}')\n    stems = sorted(path.name[:-5] for path in TEST_DIR.iterdir() if path.name.endswith('.zarr'))\n    if not stems:\n        raise FileNotFoundError(f'No test .zarr files found in {TEST_DIR}')\n    return stems\n\n\ntest_stems = list_test_stems()\nprint('target test videos:', len(test_stems))\nassert set(test_stems) == set(sample_ids), 'Pseudo-test stems do not match selected sample_ids.'\n\nsplits_path = REPO_DIR / 'kaggle_test_splits_50ep.json'\nsplits_path.parent.mkdir(parents=True, exist_ok=True)\nsplits_path.write_text(json.dumps([{'split': 0, 'train': [], 'val': [], 'test': test_stems}], indent=2))\n\npred_dir = REPO_DIR / 'predictions' / 'unknown' / METHOD / 'split_0'\nif CLEAR_OLD_PREDICTIONS and pred_dir.exists():\n    shutil.rmtree(pred_dir)\npred_dir.mkdir(parents=True, exist_ok=True)\n\nprint('restoring checkpoint geffs for selected samples:', len(sample_ids))\nfor sample_id in tqdm(test_stems, desc='restore targeted checkpoint'):\n    src = CHECKPOINT_DIR / f'{sample_id}.geff'\n    dst = pred_dir / src.name\n    if dst.exists():\n        continue\n    try:\n        os.symlink(src, dst, target_is_directory=True)\n    except Exception:\n        shutil.copytree(src, dst)\n\nrestored = sorted(pred_dir.glob('*.geff'))\nprint('restored predictions:', len(restored))\nassert len(restored) == len(test_stems), 'Restored prediction count does not match target stems.'\n\n# Build cell expects predict_seconds for run_stats; prediction is reused from checkpoint here.\npredict_seconds = 0.0\n"
BUILD_SUBMISSION_CODE = 'import tracksdata as td\nimport numpy as np\nimport blosc2\nimport torch\nfrom scipy.optimize import linear_sum_assignment\n\nSUBMISSION_COLUMNS = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]\nCSV_COLUMNS = ["id", *SUBMISSION_COLUMNS]\nVOXEL_SCALE_UM = (1.625, 0.40625, 0.40625)\n\n\ndef graph_from_geff(path: Path):\n    graph = td.graph.IndexedRXGraph.from_geff(path)\n    return graph[0] if isinstance(graph, tuple) else graph\n\n\ndef edge_distance_um(source: dict[str, object], target: dict[str, object]) -> float:\n    dz = (float(source["z"]) - float(target["z"])) * VOXEL_SCALE_UM[0]\n    dy = (float(source["y"]) - float(target["y"])) * VOXEL_SCALE_UM[1]\n    dx = (float(source["x"]) - float(target["x"])) * VOXEL_SCALE_UM[2]\n    return math.sqrt(dz * dz + dy * dy + dx * dx)\n\n\ndef point_distance_um(a: tuple[float, float, float], b: tuple[float, float, float]) -> float:\n    dz = (a[0] - b[0]) * VOXEL_SCALE_UM[0]\n    dy = (a[1] - b[1]) * VOXEL_SCALE_UM[1]\n    dx = (a[2] - b[2]) * VOXEL_SCALE_UM[2]\n    return math.sqrt(dz * dz + dy * dy + dx * dx)\n\n\ndef node_point(node: dict[str, object]) -> tuple[float, float, float]:\n    return (float(node["z"]), float(node["y"]), float(node["x"]))\n\n\ndef edge_sort_key(edge: dict[str, object]) -> tuple[float, float]:\n    prob = edge.get("edge_prob")\n    prob_value = float(prob) if prob is not None else 0.0\n    return prob_value, -float(edge["distance_um"])\n\n\ndef _next_node_id(nodes_by_id: dict[int, dict[str, object]]) -> int:\n    return max(nodes_by_id) + 1 if nodes_by_id else 1\n\n\ndef _ff_pool_frame_xy(volume: np.ndarray, factor: int) -> np.ndarray:\n    if factor <= 1:\n        return volume.astype(np.float32, copy=False)\n    z, y, x = volume.shape\n    y2 = (y // factor) * factor\n    x2 = (x // factor) * factor\n    cropped = volume[:, :y2, :x2].astype(np.float32, copy=False)\n    return cropped.reshape(z, y2 // factor, factor, x2 // factor, factor).mean(axis=(2, 4))\n\n\ndef _ff_normalize_dynamic_range(volume: np.ndarray, cfg: object) -> np.ndarray:\n    vol = np.asarray(volume, dtype=np.float32)\n    lo = float(np.percentile(vol, float(getattr(cfg, "norm_lo_pct", 50.0))))\n    hi = float(np.percentile(vol, float(getattr(cfg, "norm_hi_pct", 99.5))))\n    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:\n        return np.zeros_like(vol, dtype=np.float32)\n    ratio = (vol - lo) / (hi - lo)\n    return np.clip(\n        ratio,\n        float(getattr(cfg, "norm_clip_lo", -0.5)),\n        float(getattr(cfg, "norm_clip_hi", 6.0)),\n    ).astype(np.float32)\n\n\ndef _ff_find_peaks(heatmap: np.ndarray, threshold: float, min_distance: int) -> np.ndarray:\n    try:\n        from skimage.feature import peak_local_max\n        return peak_local_max(\n            heatmap,\n            min_distance=int(min_distance),\n            threshold_abs=float(threshold),\n            exclude_border=False,\n        ).astype(np.int64)\n    except Exception:\n        from scipy.ndimage import maximum_filter\n        size = 2 * int(min_distance) + 1\n        local = maximum_filter(heatmap, size=size, mode="nearest")\n        peaks = np.argwhere((heatmap == local) & (heatmap >= float(threshold)))\n        if len(peaks) == 0:\n            return np.empty((0, 3), dtype=np.int64)\n        values = heatmap[peaks[:, 0], peaks[:, 1], peaks[:, 2]]\n        return peaks[np.argsort(-values)].astype(np.int64)\n\n\ndef _ff_physical_nms(coords: np.ndarray, scores: np.ndarray, radius_um: float) -> tuple[np.ndarray, np.ndarray]:\n    if len(coords) == 0:\n        return coords, scores\n    scale = np.asarray(VOXEL_SCALE_UM, dtype=np.float64)\n    points = coords.astype(np.float64) * scale[None, :]\n    order = np.argsort(-scores)\n    keep: list[int] = []\n    suppressed = np.zeros(len(coords), dtype=bool)\n    radius2 = float(radius_um) ** 2\n    for idx in order:\n        if suppressed[idx]:\n            continue\n        keep.append(int(idx))\n        d2 = np.sum((points - points[idx]) ** 2, axis=1)\n        suppressed[d2 <= radius2] = True\n    keep_arr = np.asarray(keep, dtype=np.int64)\n    return coords[keep_arr], scores[keep_arr]\n\n\ndef _ff_refine_centroid(volume: np.ndarray, coord: np.ndarray) -> np.ndarray:\n    if not FULL_FRAME_REFINE_CENTROIDS:\n        return coord.astype(np.float64, copy=False)\n    zc, yc, xc = [int(round(float(v))) for v in coord]\n    z0, z1 = max(0, zc - 2), min(volume.shape[0], zc + 3)\n    y0, y1 = max(0, yc - 5), min(volume.shape[1], yc + 6)\n    x0, x1 = max(0, xc - 5), min(volume.shape[2], xc + 6)\n    patch = volume[z0:z1, y0:y1, x0:x1].astype(np.float32, copy=False)\n    if patch.size == 0:\n        return coord.astype(np.float64, copy=False)\n    background = float(np.percentile(patch, 20.0))\n    weights = np.maximum(patch - background, 0.0)\n    total = float(weights.sum())\n    if total <= 1e-6:\n        local = np.unravel_index(int(np.argmax(patch)), patch.shape)\n        return np.asarray([z0 + local[0], y0 + local[1], x0 + local[2]], dtype=np.float64)\n    zz, yy, xx = np.indices(patch.shape, dtype=np.float32)\n    return np.asarray([\n        z0 + float((zz * weights).sum() / total),\n        y0 + float((yy * weights).sum() / total),\n        x0 + float((xx * weights).sum() / total),\n    ], dtype=np.float64)\n\n\n\ndef _ff_load_gate_summary(checkpoint_path: Path, artifacts: Path) -> dict:\n    manifest = artifact_manifest(artifacts)\n    candidates: list[Path] = [\n        checkpoint_path.parent / "gate_summary.json",\n        artifacts / "weights" / "full_frame_center" / "gate_summary.json",\n    ]\n    for section in [\n        manifest.get("models", {}).get("full_frame_center", {}),\n        manifest.get("full_frame_center", {}),\n    ]:\n        diagnostics = section.get("diagnostics", {}) if isinstance(section, dict) else {}\n        entry = diagnostics.get("gate_summary.json") if isinstance(diagnostics, dict) else None\n        rel = entry.get("path") if isinstance(entry, dict) else None\n        if rel:\n            candidates.append(artifacts / str(rel))\n    seen: set[Path] = set()\n    for path in candidates:\n        try:\n            key = path.resolve() if path.exists() else path\n        except Exception:\n            key = path\n        if key in seen:\n            continue\n        seen.add(key)\n        if path.exists():\n            try:\n                return json.loads(path.read_text())\n            except Exception as exc:\n                print("Could not read full-frame gate summary:", path, type(exc).__name__, exc)\n    return {}\n\n\ndef _ff_choose_peak_threshold(gate_summary: dict, default_threshold: float) -> tuple[float, str]:\n    if not FULL_FRAME_USE_GATE_CALIBRATION:\n        return float(default_threshold), "configured_default"\n    rows = gate_summary.get("threshold_metrics", []) if isinstance(gate_summary, dict) else []\n    usable = []\n    for row in rows:\n        try:\n            threshold = float(row["threshold"])\n            precision = float(row.get("precision_sparse", float("nan")))\n            pred_per_frame = float(row.get("pred_per_frame", float("inf")))\n        except Exception:\n            continue\n        if not np.isfinite(threshold) or not np.isfinite(precision):\n            continue\n        if precision < FULL_FRAME_GATE_TARGET_PRECISION:\n            continue\n        if FULL_FRAME_GATE_MAX_PRED_PER_FRAME > 0 and pred_per_frame > FULL_FRAME_GATE_MAX_PRED_PER_FRAME:\n            continue\n        usable.append((threshold, precision, pred_per_frame))\n    if usable:\n        # Lowest threshold satisfying the gate keeps recall while respecting the sparse-label precision floor.\n        threshold, precision, pred_per_frame = sorted(usable, key=lambda x: x[0])[0]\n        return float(threshold), f"gate_summary_precision>={FULL_FRAME_GATE_TARGET_PRECISION:g};precision={precision:.4f};pred_per_frame={pred_per_frame:.2f}"\n    return float(default_threshold), "gate_summary_no_feasible_threshold"\n\n\ndef _ff_manifest_weight_paths(manifest_path: Path) -> list[Path]:\n    if not manifest_path.exists():\n        return []\n    try:\n        manifest = json.loads(manifest_path.read_text())\n    except Exception as exc:\n        print("Could not read full-frame center manifest:", manifest_path, type(exc).__name__, exc)\n        return []\n\n    root = manifest_path.parent\n    sections: list[dict[str, object]] = []\n    for section in [\n        manifest.get("model", {}),\n        manifest.get("models", {}).get("full_frame_center", {}) if isinstance(manifest.get("models", {}), dict) else {},\n        manifest.get("full_frame_center", {}),\n    ]:\n        if isinstance(section, dict):\n            sections.append(section)\n\n    prefer_last = "checkpoint_last.pt" in str(FULL_FRAME_CHECKPOINT_DEFAULT)\n    checkpoint_keys = ("last_checkpoint", "best_checkpoint") if prefer_last else ("best_checkpoint", "last_checkpoint")\n    direct_names = ("checkpoint_last.pt", "best.pt", "last.pt") if prefer_last else ("best.pt", "checkpoint_last.pt", "last.pt")\n\n    candidates: list[Path] = []\n    for section in sections:\n        for key in ("weight_path", "path"):\n            rel = section.get(key)\n            if isinstance(rel, str) and rel:\n                candidates.append(root / rel)\n        for key in checkpoint_keys:\n            item = section.get(key)\n            if isinstance(item, dict):\n                rel = item.get("path")\n                if isinstance(rel, str) and rel:\n                    candidates.append(root / rel)\n\n    for name in direct_names:\n        candidates.append(root / "weights" / "full_frame_center" / name)\n        candidates.append(root / name)\n    candidates.append(root / FULL_FRAME_CENTER_RELATIVE)\n    return candidates\n\n\ndef _ff_checkpoint_candidates(artifacts: Path) -> list[Path]:\n    candidates: list[Path] = []\n    explicit = os.environ.get("BIOHUB_FULL_FRAME_CHECKPOINT", FULL_FRAME_CHECKPOINT_DEFAULT).strip()\n    prefer_last_default = "checkpoint_last.pt" in str(FULL_FRAME_CHECKPOINT_DEFAULT)\n    if explicit:\n        explicit_path = Path(explicit)\n        if prefer_last_default and not explicit_path.exists():\n            raise FileNotFoundError(\n                "DeepCenter checkpoint_last.pt was requested but the explicit checkpoint path does not exist: "\n                f"{explicit_path}. Attach the intended DeepCenter dataset version or set BIOHUB_FULL_FRAME_CHECKPOINT."\n            )\n        candidates.append(explicit_path)\n\n    manifest_explicit = os.environ.get("BIOHUB_FULL_FRAME_MANIFEST", FULL_FRAME_MANIFEST_DEFAULT).strip()\n    if manifest_explicit:\n        candidates.extend(_ff_manifest_weight_paths(Path(manifest_explicit)))\n\n    manifest = artifact_manifest(artifacts)\n    for section in [\n        manifest.get("models", {}).get("full_frame_center", {}),\n        manifest.get("full_frame_center", {}),\n    ]:\n        if not isinstance(section, dict):\n            continue\n        for key in ("weight_path", "path"):\n            rel = section.get(key)\n            if isinstance(rel, str) and rel:\n                candidates.append(artifacts / rel)\n        for key in ("best_checkpoint", "last_checkpoint"):\n            item = section.get(key)\n            if isinstance(item, dict):\n                rel = item.get("path")\n                if isinstance(rel, str) and rel:\n                    candidates.append(artifacts / rel)\n\n    prefer_last = "checkpoint_last.pt" in str(FULL_FRAME_CHECKPOINT_DEFAULT)\n    fallback_names = ("checkpoint_last.pt", "best.pt", "last.pt") if prefer_last else ("best.pt", "checkpoint_last.pt", "last.pt")\n    for name in fallback_names:\n        candidates.append(artifacts / "weights" / "full_frame_center" / name)\n        candidates.append(artifacts / name)\n    candidates.extend([\n        artifacts / FULL_FRAME_CENTER_RELATIVE,\n        Path("PublicNotebook") / "best.pt",\n        Path("notebooks") / "public_examples" / "early_baselines" / "best.pt",\n    ])\n\n    input_root = Path("/kaggle/input")\n    if input_root.exists():\n        preferred_dirs = [\n            input_root / "datasets" / "pilkwang" / "biohub-deepcenter-unet3d-center-prior-v1",\n            input_root / "biohub-deepcenter-unet3d-center-prior-v1",\n            input_root / "zebracelltrace-ff-checkpoint",\n            input_root / "datasets" / "pilkwang" / "zebracelltrace-ff-checkpoint",\n        ]\n        name_order = ("checkpoint_last.pt", "best.pt", "last.pt") if "checkpoint_last.pt" in str(FULL_FRAME_CHECKPOINT_DEFAULT) else ("best.pt", "checkpoint_last.pt", "last.pt")\n        for directory in preferred_dirs:\n            candidates.extend(_ff_manifest_weight_paths(directory / "ARTIFACT_MANIFEST.json"))\n            for name in name_order:\n                candidates.append(directory / name)\n                candidates.append(directory / "weights" / "full_frame_center" / name)\n        for name in name_order:\n            candidates.extend(sorted(input_root.glob(f"**/full_frame_center/**/{name}")))\n            candidates.extend(sorted(input_root.glob(f"**/{name}")))\n\n    seen: set[Path] = set()\n    out: list[Path] = []\n    for path in candidates:\n        path = path.expanduser()\n        try:\n            key = path.resolve() if path.exists() else path\n        except Exception:\n            key = path\n        if key in seen:\n            continue\n        seen.add(key)\n        out.append(path)\n    return out\n\n\nclass _FFConvBlock3d(torch.nn.Module):\n    def __init__(self, in_channels: int, out_channels: int) -> None:\n        super().__init__()\n        groups = min(8, out_channels)\n        self.block = torch.nn.Sequential(\n            torch.nn.Conv3d(in_channels, out_channels, 3, padding=1, bias=False),\n            torch.nn.GroupNorm(groups, out_channels),\n            torch.nn.SiLU(inplace=True),\n            torch.nn.Conv3d(out_channels, out_channels, 3, padding=1, bias=False),\n            torch.nn.GroupNorm(groups, out_channels),\n            torch.nn.SiLU(inplace=True),\n        )\n\n    def forward(self, x):\n        return self.block(x)\n\n\nclass _FFDeepCenterUNet3D(torch.nn.Module):\n    def __init__(self, in_channels: int = 1, base_channels: int = 24) -> None:\n        super().__init__()\n        c = int(base_channels)\n        self.enc1 = _FFConvBlock3d(in_channels, c)\n        self.down1 = torch.nn.MaxPool3d(2, 2)\n        self.enc2 = _FFConvBlock3d(c, c * 2)\n        self.down2 = torch.nn.MaxPool3d(2, 2)\n        self.enc3 = _FFConvBlock3d(c * 2, c * 4)\n        self.down3 = torch.nn.MaxPool3d(2, 2)\n        self.bottleneck = _FFConvBlock3d(c * 4, c * 8)\n        self.up3 = torch.nn.ConvTranspose3d(c * 8, c * 4, 2, 2)\n        self.dec3 = _FFConvBlock3d(c * 8, c * 4)\n        self.up2 = torch.nn.ConvTranspose3d(c * 4, c * 2, 2, 2)\n        self.dec2 = _FFConvBlock3d(c * 4, c * 2)\n        self.up1 = torch.nn.ConvTranspose3d(c * 2, c, 2, 2)\n        self.dec1 = _FFConvBlock3d(c * 2, c)\n        self.head = torch.nn.Conv3d(c, 1, 1)\n\n    def forward(self, x):\n        e1 = self.enc1(x)\n        e2 = self.enc2(self.down1(e1))\n        e3 = self.enc3(self.down2(e2))\n        b = self.bottleneck(self.down3(e3))\n        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))\n        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))\n        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))\n        return self.head(d1)\n\n\ndef load_full_frame_detector():\n    if not USE_FULL_FRAME_CENTER_FUSION:\n        print("Full-frame center fusion disabled by configuration.")\n        return None\n    try:\n        import torch\n        from types import SimpleNamespace\n    except Exception as exc:\n        if REQUIRE_FULL_FRAME_CENTER:\n            raise\n        print("Full-frame center fusion unavailable; torch import failed:", exc)\n        return None\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    load_errors: list[str] = []\n    for checkpoint_path in _ff_checkpoint_candidates(ARTIFACTS):\n        if not checkpoint_path.exists():\n            continue\n        try:\n            print("Trying full-frame center checkpoint:", checkpoint_path)\n            checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)\n            if not isinstance(checkpoint, dict) or "model_state" not in checkpoint:\n                raise ValueError("checkpoint has no model_state")\n            cfg = SimpleNamespace(**checkpoint.get("config", {}))\n            model = _FFDeepCenterUNet3D(base_channels=int(getattr(cfg, "base_channels", 24)))\n            model.load_state_dict(checkpoint["model_state"])\n            model.to(device)\n            model.eval()\n            gate_summary = _ff_load_gate_summary(checkpoint_path, ARTIFACTS)\n            peak_threshold, threshold_source = _ff_choose_peak_threshold(gate_summary, FULL_FRAME_PEAK_THRESHOLD)\n            print("Loaded full-frame checkpoint:", checkpoint_path)\n            print("Full-frame checkpoint epoch:", checkpoint.get("epoch"), "best_score:", checkpoint.get("best_score"))\n            print("Full-frame peak threshold:", peak_threshold, "source:", threshold_source)\n            return {\n                "model": model,\n                "cfg": cfg,\n                "device": device,\n                "path": checkpoint_path,\n                "gate_summary": gate_summary,\n                "peak_threshold": float(peak_threshold),\n                "threshold_source": threshold_source,\n            }\n        except Exception as exc:\n            load_errors.append(f"{checkpoint_path}: {type(exc).__name__}: {exc}")\n            print("Skipping incompatible full-frame checkpoint:", checkpoint_path, "|", type(exc).__name__, exc)\n            continue\n    message = "No full-frame center checkpoint found; fusion path will be skipped."\n    if REQUIRE_FULL_FRAME_CENTER:\n        checked = "\\n".join(str(p) for p in _ff_checkpoint_candidates(ARTIFACTS)[:80])\n        errors = "\\n".join(load_errors[-20:])\n        raise FileNotFoundError(message + "\\nChecked:\\n" + checked + ("\\nLoad errors:\\n" + errors if errors else ""))\n    print(message)\n    return None\n\n\n@torch.no_grad()\ndef detect_full_frame_centers(dataset: str, detector_bundle: dict | None) -> dict[int, list[tuple[float, float, float, float]]]:\n    if detector_bundle is None:\n        return {}\n    model = detector_bundle["model"]\n    cfg = detector_bundle["cfg"]\n    device = detector_bundle["device"]\n    zarr_path = TEST_DIR / f"{dataset}.zarr"\n    meta = json.loads((zarr_path / "0" / "zarr.json").read_text())\n    shape = tuple(int(v) for v in meta["shape"])\n    pool_factor = int(getattr(cfg, "pool_factor", 4))\n    frame_cache: dict[int, np.ndarray] = {}\n    out: dict[int, list[tuple[float, float, float, float]]] = {}\n    for t in range(shape[0]):\n        volume = read_test_frame(dataset, t, frame_cache)\n        pooled = _ff_pool_frame_xy(volume, pool_factor)\n        image = _ff_normalize_dynamic_range(pooled, cfg)\n        tensor = torch.from_numpy(image[None, None, ...]).to(device=device, dtype=torch.float32)\n        heatmap = torch.sigmoid(model(tensor))[0, 0].detach().cpu().numpy()\n        peak_threshold = float(detector_bundle.get("peak_threshold", FULL_FRAME_PEAK_THRESHOLD))\n        peaks = _ff_find_peaks(heatmap, peak_threshold, FULL_FRAME_MIN_PEAK_DISTANCE)\n        if len(peaks) == 0:\n            out[t] = []\n            frame_cache.pop(t, None)\n            continue\n        scores = heatmap[peaks[:, 0], peaks[:, 1], peaks[:, 2]].astype(np.float64)\n        coords = peaks.astype(np.float64)\n        coords[:, 1] = coords[:, 1] * pool_factor + (pool_factor - 1) / 2.0\n        coords[:, 2] = coords[:, 2] * pool_factor + (pool_factor - 1) / 2.0\n        refined = []\n        for coord in coords:\n            point = _ff_refine_centroid(volume, coord)\n            point[0] = np.clip(point[0], 0, shape[1] - 1)\n            point[1] = np.clip(point[1], 0, shape[2] - 1)\n            point[2] = np.clip(point[2], 0, shape[3] - 1)\n            refined.append(point)\n        coords = np.asarray(refined, dtype=np.float64)\n        coords, scores = _ff_physical_nms(coords, scores, FULL_FRAME_NMS_RADIUS_UM)\n        order = np.argsort(-scores)\n        out[t] = [(float(coords[i, 0]), float(coords[i, 1]), float(coords[i, 2]), float(scores[i])) for i in order]\n        frame_cache.pop(t, None)\n    return out\n\n\ndef fuse_full_frame_nodes(\n    dataset: str,\n    nodes_by_id: dict[int, dict[str, object]],\n    detector_bundle: dict | None,\n) -> dict[str, int | float]:\n    stats: dict[str, int | float] = {\n        "full_frame_enabled": int(detector_bundle is not None),\n        "full_frame_peak_threshold": float(detector_bundle.get("peak_threshold", FULL_FRAME_PEAK_THRESHOLD)) if detector_bundle is not None else float(FULL_FRAME_PEAK_THRESHOLD),\n        "full_frame_candidates": 0,\n        "full_frame_added_nodes": 0,\n        "full_frame_skipped_existing": 0,\n        "full_frame_skipped_cap": 0,\n    }\n    if detector_bundle is None:\n        return stats\n    candidates_by_t = detect_full_frame_centers(dataset, detector_bundle)\n    ids_by_t: dict[int, list[int]] = {}\n    for node_id, node in nodes_by_id.items():\n        ids_by_t.setdefault(int(node["t"]), []).append(node_id)\n    next_id = _next_node_id(nodes_by_id)\n    for t, candidates in candidates_by_t.items():\n        stats["full_frame_candidates"] += len(candidates)\n        existing_ids = ids_by_t.setdefault(t, [])\n        existing_points = [node_point(nodes_by_id[node_id]) for node_id in existing_ids]\n        if FULL_FRAME_MAX_ADDED_FRAC > 0:\n            frac_cap = max(1, int(math.ceil(max(len(existing_ids), 1) * FULL_FRAME_MAX_ADDED_FRAC)))\n        else:\n            frac_cap = 0\n        absolute_cap = max(0, int(FULL_FRAME_MAX_ADDED_ABS))\n        if absolute_cap == 0 or frac_cap == 0:\n            max_added = 0\n        else:\n            max_added = min(len(candidates), absolute_cap, frac_cap)\n        added_this_frame = 0\n        for z, y, x, score in candidates:\n            if added_this_frame >= max_added:\n                stats["full_frame_skipped_cap"] += 1\n                continue\n            point = (z, y, x)\n            if existing_points:\n                nearest = min(point_distance_um(point, other) for other in existing_points)\n                if nearest < FULL_FRAME_EXISTING_GATE_UM:\n                    stats["full_frame_skipped_existing"] += 1\n                    continue\n            node_id = next_id\n            next_id += 1\n            nodes_by_id[node_id] = {\n                "node_id": node_id,\n                "t": int(t),\n                "z": float(z),\n                "y": float(y),\n                "x": float(x),\n                "full_frame_score": float(score),\n            }\n            existing_ids.append(node_id)\n            existing_points.append(point)\n            added_this_frame += 1\n            stats["full_frame_added_nodes"] += 1\n    return stats\n\n\n\ndef read_test_frame(dataset: str, t: int, frame_cache: dict[int, np.ndarray]) -> np.ndarray:\n    if t in frame_cache:\n        return frame_cache[t]\n    zarr_path = TEST_DIR / f"{dataset}.zarr"\n    meta = json.loads((zarr_path / "0" / "zarr.json").read_text())\n    shape = tuple(int(v) for v in meta["shape"])\n    dtype = np.dtype(meta["data_type"])\n    frame_shape = shape[1:]\n    chunk_path = zarr_path / "0" / "c" / str(t) / "0" / "0" / "0"\n    try:\n        raw = chunk_path.read_bytes()\n        arr = np.frombuffer(blosc2.decompress(raw), dtype=dtype)\n        if arr.size == int(np.prod(frame_shape)):\n            frame = arr.reshape(frame_shape).copy()\n            frame_cache[t] = frame\n            return frame\n    except Exception:\n        pass\n    import zarr\n    frame = np.asarray(zarr.open(zarr_path / "0", mode="r")[t])\n    frame_cache[t] = frame\n    return frame\n\n\ndef refine_synthetic_midpoint(\n    dataset: str | None,\n    t: int,\n    midpoint: tuple[float, float, float],\n    frame_cache: dict[int, np.ndarray],\n    stats: dict[str, int],\n) -> tuple[float, float, float]:\n    if not GAP_REFINE_SYNTHETIC or dataset is None:\n        return midpoint\n    try:\n        frame = read_test_frame(dataset, t, frame_cache)\n        z, y, x = [int(round(v)) for v in midpoint]\n        z0 = max(0, z - GAP_REFINE_WIN_Z)\n        z1 = min(frame.shape[0], z + GAP_REFINE_WIN_Z + 1)\n        y0 = max(0, y - GAP_REFINE_WIN_YX)\n        y1 = min(frame.shape[1], y + GAP_REFINE_WIN_YX + 1)\n        x0 = max(0, x - GAP_REFINE_WIN_YX)\n        x1 = min(frame.shape[2], x + GAP_REFINE_WIN_YX + 1)\n        patch = frame[z0:z1, y0:y1, x0:x1].astype(np.float64)\n        if patch.size == 0:\n            stats["gap_refine_failed"] += 1\n            return midpoint\n        baseline = float(np.percentile(patch, 20.0))\n        weights = np.maximum(patch - baseline, 0.0)\n        total = float(weights.sum())\n        if total <= 0:\n            stats["gap_refine_failed"] += 1\n            return midpoint\n        zz = np.arange(z0, z1, dtype=np.float64)[:, None, None]\n        yy = np.arange(y0, y1, dtype=np.float64)[None, :, None]\n        xx = np.arange(x0, x1, dtype=np.float64)[None, None, :]\n        refined = (\n            float((weights * zz).sum() / total),\n            float((weights * yy).sum() / total),\n            float((weights * xx).sum() / total),\n        )\n        if point_distance_um(refined, midpoint) > GAP_REFINE_MAX_SHIFT_UM:\n            stats["gap_refine_rejected_shift"] += 1\n            return midpoint\n        stats["gap_refined_synthetic"] += 1\n        return refined\n    except Exception:\n        stats["gap_refine_failed"] += 1\n        return midpoint\n\n\ndef _position_um(node: dict[str, object]) -> np.ndarray:\n    return np.array(\n        [float(node["z"]) * VOXEL_SCALE_UM[0], float(node["y"]) * VOXEL_SCALE_UM[1], float(node["x"]) * VOXEL_SCALE_UM[2]],\n        dtype=np.float64,\n    )\n\n\ndef motion_relink_edges(\n    nodes_by_id: dict[int, dict[str, object]],\n    stats: dict[str, int],\n    learned_edge_probs: dict[tuple[int, int], float] | None = None,\n) -> list[dict[str, object]]:\n    if not OUTPUT_MOTION_RELINK or not nodes_by_id:\n        return []\n\n    learned_edge_probs = learned_edge_probs or {}\n\n    def learned_prob(source_id: int, target_id: int) -> float:\n        value = learned_edge_probs.get((source_id, target_id), 0.0)\n        try:\n            value = float(value)\n        except (TypeError, ValueError):\n            return 0.0\n        if not np.isfinite(value):\n            return 0.0\n        if value < 0.0 or value > 1.0:\n            value = 1.0 / (1.0 + math.exp(-max(-20.0, min(20.0, value))))\n        return float(np.clip(value, 0.0, 1.0))\n\n    ids_by_t: dict[int, list[int]] = {}\n    for node_id, node in nodes_by_id.items():\n        ids_by_t.setdefault(int(node["t"]), []).append(node_id)\n    for ids in ids_by_t.values():\n        ids.sort()\n\n    frame_sizes = [len(ids) for ids in ids_by_t.values()]\n    if frame_sizes and max(frame_sizes) > MOTION_RELINK_MAX_FRAME_NODES:\n        stats["motion_relink_skipped_large_frame"] = 1\n        return []\n\n    position_um = {node_id: _position_um(node) for node_id, node in nodes_by_id.items()}\n    predecessor_position_um: dict[int, np.ndarray] = {}\n    selected_edges: list[dict[str, object]] = []\n\n    def assign_pass(\n        source_ids: list[int],\n        target_ids: list[int],\n        gate_um: float,\n    ) -> list[tuple[int, int, float, float, float]]:\n        if not source_ids or not target_ids:\n            return []\n        big = gate_um * 1000.0 + 1.0\n        cost = np.full((len(source_ids), len(target_ids)), big, dtype=np.float64)\n        raw_dist = np.full_like(cost, np.inf)\n        motion_dist = np.full_like(cost, np.inf)\n        prob_matrix = np.zeros_like(cost)\n        for i, source_id in enumerate(source_ids):\n            source_pos = position_um[source_id]\n            prev_pos = predecessor_position_um.get(source_id)\n            if prev_pos is None:\n                predicted = source_pos\n            else:\n                predicted = source_pos + MOTION_RELINK_VELOCITY_WEIGHT * (source_pos - prev_pos)\n            for j, target_id in enumerate(target_ids):\n                target_pos = position_um[target_id]\n                raw = float(np.linalg.norm(target_pos - source_pos))\n                if raw > gate_um:\n                    continue\n                motion = float(np.linalg.norm(target_pos - predicted))\n                prob = learned_prob(source_id, target_id)\n                raw_dist[i, j] = raw\n                motion_dist[i, j] = motion\n                prob_matrix[i, j] = prob\n                cost[i, j] = motion + 0.05 * raw - MOTION_RELINK_LEARNED_BONUS * prob\n        row_ind, col_ind = linear_sum_assignment(cost)\n        matches: list[tuple[int, int, float, float, float]] = []\n        for r, c in zip(row_ind, col_ind):\n            if cost[r, c] >= big:\n                continue\n            matches.append((\n                source_ids[int(r)],\n                target_ids[int(c)],\n                float(raw_dist[r, c]),\n                float(motion_dist[r, c]),\n                float(prob_matrix[r, c]),\n            ))\n        return matches\n\n    times = sorted(ids_by_t)\n    for t in times:\n        source_ids = ids_by_t.get(t, [])\n        target_ids = ids_by_t.get(t + 1, [])\n        if not source_ids or not target_ids:\n            continue\n        unmatched_sources = set(source_ids)\n        unmatched_targets = set(target_ids)\n        frame_matches: list[tuple[int, int, float, float, str, float]] = []\n        for pass_name, gate_um in (("tight", MOTION_RELINK_TIGHT_UM), ("relaxed", MOTION_RELINK_RELAXED_UM)):\n            pass_sources = [node_id for node_id in source_ids if node_id in unmatched_sources]\n            pass_targets = [node_id for node_id in target_ids if node_id in unmatched_targets]\n            matches = assign_pass(pass_sources, pass_targets, gate_um)\n            for source_id, target_id, raw, motion, prob in matches:\n                if source_id not in unmatched_sources or target_id not in unmatched_targets:\n                    continue\n                unmatched_sources.remove(source_id)\n                unmatched_targets.remove(target_id)\n                frame_matches.append((source_id, target_id, raw, motion, pass_name, prob))\n                if pass_name == "tight":\n                    stats["motion_relink_tight_edges"] += 1\n                else:\n                    stats["motion_relink_relaxed_edges"] += 1\n        for source_id, target_id, raw, motion, pass_name, prob in frame_matches:\n            selected_edges.append({\n                "source_id": source_id,\n                "target_id": target_id,\n                "edge_prob": prob,\n                "distance_um": raw,\n                "motion_distance_um": motion,\n                "motion_relinked": 1,\n                "motion_pass": pass_name,\n            })\n            predecessor_position_um[target_id] = position_um[source_id]\n        stats["motion_relink_frames"] += 1\n\n    stats["motion_relink_edges"] = len(selected_edges)\n    return selected_edges\n\ndef close_single_frame_gaps(\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n    dataset: str | None = None,\n) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:\n    if not OUTPUT_GAP_CLOSE or GAP_CLOSE_MAX_GAP < 1 or not edges:\n        return nodes_by_id, edges\n\n    outgoing = {int(edge["source_id"]) for edge in edges}\n    incoming = {int(edge["target_id"]) for edge in edges}\n    incident = outgoing | incoming\n\n    ends_by_t: dict[int, list[int]] = {}\n    starts_by_t: dict[int, list[int]] = {}\n    isolated_by_t: dict[int, list[int]] = {}\n    for node_id, node in nodes_by_id.items():\n        t = int(node["t"])\n        if node_id not in outgoing:\n            ends_by_t.setdefault(t, []).append(node_id)\n        if node_id not in incoming:\n            starts_by_t.setdefault(t, []).append(node_id)\n        if node_id not in incident:\n            isolated_by_t.setdefault(t, []).append(node_id)\n\n    max_synthetic = min(\n        GAP_CLOSE_MAX_ADDED_ABS,\n        max(1, int(round(len(nodes_by_id) * GAP_CLOSE_MAX_ADDED_FRAC))) if GAP_CLOSE_MAX_ADDED_FRAC > 0 else 0,\n    )\n    next_id = _next_node_id(nodes_by_id)\n    frame_cache: dict[int, np.ndarray] = {}\n    used_starts: set[int] = set()\n    used_isolated: set[int] = set()\n    synthetic_added = 0\n    new_edges: list[dict[str, object]] = []\n\n    effective_gap_max = min(GAP_CLOSE_MAX_GAP, 1)\n    stats["gap_close_effective_max_gap"] = effective_gap_max\n    for gap in range(1, effective_gap_max + 1):\n        for t, end_ids in sorted(ends_by_t.items()):\n            start_ids = [sid for sid in starts_by_t.get(t + gap + 1, []) if sid not in used_starts]\n            if not end_ids or not start_ids:\n                continue\n\n            end_points = [node_point(nodes_by_id[eid]) for eid in end_ids]\n            start_points = [node_point(nodes_by_id[sid]) for sid in start_ids]\n            threshold_um = GAP_CLOSE_UM * (gap + 1)\n            d = np.zeros((len(end_ids), len(start_ids)), dtype=np.float64)\n            for i, ep in enumerate(end_points):\n                for j, sp in enumerate(start_points):\n                    d[i, j] = point_distance_um(ep, sp)\n            stats["gap_candidates"] += int((d <= threshold_um).sum())\n            if not np.isfinite(d).any():\n                continue\n\n            big = threshold_um * 1000.0 + 1.0\n            cost = np.where(d <= threshold_um, d, big)\n            row_ind, col_ind = linear_sum_assignment(cost)\n            for r, c in zip(row_ind, col_ind):\n                if d[r, c] > threshold_um:\n                    continue\n                source_id = end_ids[int(r)]\n                target_id = start_ids[int(c)]\n                if source_id in outgoing or target_id in used_starts:\n                    continue\n\n                source = nodes_by_id[source_id]\n                target = nodes_by_id[target_id]\n                mid_t = int(source["t"]) + gap\n                mid_point = (\n                    (float(source["z"]) + float(target["z"])) / 2.0,\n                    (float(source["y"]) + float(target["y"])) / 2.0,\n                    (float(source["x"]) + float(target["x"])) / 2.0,\n                )\n\n                middle_id: int | None = None\n                if GAP_CLOSE_REUSE_EXISTING:\n                    candidates = [nid for nid in isolated_by_t.get(mid_t, []) if nid not in used_isolated]\n                    if candidates:\n                        distances = [point_distance_um(node_point(nodes_by_id[nid]), mid_point) for nid in candidates]\n                        best_idx = int(np.argmin(distances))\n                        if distances[best_idx] <= GAP_CLOSE_REUSE_UM:\n                            middle_id = candidates[best_idx]\n                            used_isolated.add(middle_id)\n                            stats["gap_reused_existing"] += 1\n\n                if middle_id is None:\n                    if synthetic_added >= max_synthetic:\n                        stats["gap_skipped_node_cap"] += 1\n                        continue\n                    middle_id = next_id\n                    next_id += 1\n                    refined_point = refine_synthetic_midpoint(dataset, mid_t, mid_point, frame_cache, stats)\n                    nodes_by_id[middle_id] = {\n                        "node_id": middle_id,\n                        "t": mid_t,\n                        "z": refined_point[0],\n                        "y": refined_point[1],\n                        "x": refined_point[2],\n                    }\n                    synthetic_added += 1\n                    stats["gap_inserted_synthetic"] += 1\n\n                middle = nodes_by_id[middle_id]\n                e1 = {\n                    "source_id": source_id,\n                    "target_id": middle_id,\n                    "edge_prob": None,\n                    "distance_um": edge_distance_um(source, middle),\n                    "gap_closed": 1,\n                }\n                e2 = {\n                    "source_id": middle_id,\n                    "target_id": target_id,\n                    "edge_prob": None,\n                    "distance_um": edge_distance_um(middle, target),\n                    "gap_closed": 1,\n                }\n                new_edges.extend([e1, e2])\n                outgoing.add(source_id)\n                incoming.add(middle_id)\n                outgoing.add(middle_id)\n                incoming.add(target_id)\n                used_starts.add(target_id)\n                stats["gap_pairs_selected"] += 1\n                stats["gap_added_edges"] += 2\n\n    if new_edges:\n        edges = [*edges, *new_edges]\n    stats["gap_added_nodes"] = stats["gap_inserted_synthetic"]\n    return nodes_by_id, edges\n\n\ndef _single_successor_map(edges: list[dict[str, object]]) -> dict[int, int]:\n    by_source: dict[int, list[int]] = {}\n    for edge in edges:\n        by_source.setdefault(int(edge["source_id"]), []).append(int(edge["target_id"]))\n    return {source: targets[0] for source, targets in by_source.items() if len(targets) == 1}\n\n\ndef _single_predecessor_map(edges: list[dict[str, object]]) -> dict[int, int]:\n    by_target: dict[int, list[int]] = {}\n    for edge in edges:\n        by_target.setdefault(int(edge["target_id"]), []).append(int(edge["source_id"]))\n    return {target: sources[0] for target, sources in by_target.items() if len(sources) == 1}\n\n\ndef recover_strict_gap2(\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n    dataset: str | None = None,\n) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:\n    if not OUTPUT_GAP2_RECOVERY or not edges or not nodes_by_id:\n        return nodes_by_id, edges\n\n    outgoing = {int(edge["source_id"]) for edge in edges}\n    incoming = {int(edge["target_id"]) for edge in edges}\n    predecessor = _single_predecessor_map(edges)\n    successor = _single_successor_map(edges)\n\n    ends_by_t: dict[int, list[int]] = {}\n    starts_by_t: dict[int, list[int]] = {}\n    for node_id, node in nodes_by_id.items():\n        t = int(node["t"])\n        if node_id not in outgoing:\n            ends_by_t.setdefault(t, []).append(node_id)\n        if node_id not in incoming:\n            starts_by_t.setdefault(t, []).append(node_id)\n\n    cap = min(GAP2_MAX_LINKS_ABS, max(1, int(round(len(edges) * GAP2_MAX_LINKS_FRAC))))\n    proposals: list[tuple[float, int, int, int, float]] = []\n\n    def pos_um(node_id: int) -> np.ndarray:\n        node = nodes_by_id[node_id]\n        return np.array([float(node["z"]), float(node["y"]), float(node["x"])], dtype=np.float64) * np.array(VOXEL_SCALE_UM)\n\n    for t, end_ids in sorted(ends_by_t.items()):\n        start_ids = starts_by_t.get(t + 3, [])\n        if not end_ids or not start_ids:\n            continue\n        for end_id in end_ids:\n            end_pos = pos_um(end_id)\n            for start_id in start_ids:\n                start_pos = pos_um(start_id)\n                dist = float(np.linalg.norm(start_pos - end_pos))\n                if dist > GAP2_MAX_TOTAL_UM or dist / 3.0 > GAP2_MAX_STEP_UM:\n                    continue\n                step = (start_pos - end_pos) / 3.0\n                context_penalty = 0.0\n                if GAP2_REQUIRE_CONTEXT:\n                    ok_context = False\n                    prev_id = predecessor.get(end_id)\n                    if prev_id is not None:\n                        prev_step = end_pos - pos_um(prev_id)\n                        prev_norm = float(np.linalg.norm(prev_step))\n                        step_norm = float(np.linalg.norm(step))\n                        if prev_norm <= 0.01 or step_norm <= 0.01:\n                            ok_context = True\n                        else:\n                            cos = float(np.dot(prev_step, step) / (prev_norm * step_norm + 1e-9))\n                            if cos > -0.25 and np.linalg.norm(prev_step - step) <= 6.0:\n                                ok_context = True\n                            context_penalty += max(0.0, 0.25 - cos)\n                    next_id = successor.get(start_id)\n                    if next_id is not None:\n                        next_step = pos_um(next_id) - start_pos\n                        next_norm = float(np.linalg.norm(next_step))\n                        step_norm = float(np.linalg.norm(step))\n                        if next_norm <= 0.01 or step_norm <= 0.01:\n                            ok_context = True\n                        else:\n                            cos = float(np.dot(next_step, step) / (next_norm * step_norm + 1e-9))\n                            if cos > -0.25 and np.linalg.norm(next_step - step) <= 6.0:\n                                ok_context = True\n                            context_penalty += max(0.0, 0.25 - cos)\n                    if not ok_context:\n                        continue\n                proposals.append((dist + 2.0 * context_penalty, end_id, start_id, t, dist))\n\n    proposals.sort(key=lambda item: item[0])\n    stats["gap2_candidates"] = len(proposals)\n    if not proposals:\n        return nodes_by_id, edges\n\n    selected: list[tuple[float, int, int, int, float]] = []\n    used_ends: set[int] = set()\n    used_starts: set[int] = set()\n    per_frame_count: dict[int, int] = {}\n    for proposal in proposals:\n        if len(selected) >= cap:\n            stats["gap2_skipped_cap"] += 1\n            break\n        _, end_id, start_id, t, _ = proposal\n        if end_id in used_ends or start_id in used_starts:\n            continue\n        frame_cap = max(1, int(round(len(ends_by_t.get(t, [])) * GAP2_FRAME_FRAC_CAP)))\n        if per_frame_count.get(t, 0) >= frame_cap:\n            continue\n        selected.append(proposal)\n        used_ends.add(end_id)\n        used_starts.add(start_id)\n        per_frame_count[t] = per_frame_count.get(t, 0) + 1\n\n    if not selected:\n        return nodes_by_id, edges\n\n    next_node_id = _next_node_id(nodes_by_id)\n    frame_cache: dict[int, np.ndarray] = {}\n    new_edges: list[dict[str, object]] = []\n    for _, end_id, start_id, t, _ in selected:\n        source = nodes_by_id[end_id]\n        target = nodes_by_id[start_id]\n        previous_id = end_id\n        inserted_ids: list[int] = []\n        for k in (1, 2):\n            frac = k / 3.0\n            mid_t = int(source["t"]) + k\n            midpoint = (\n                float(source["z"]) + (float(target["z"]) - float(source["z"])) * frac,\n                float(source["y"]) + (float(target["y"]) - float(source["y"])) * frac,\n                float(source["x"]) + (float(target["x"]) - float(source["x"])) * frac,\n            )\n            refined_point = refine_synthetic_midpoint(dataset, mid_t, midpoint, frame_cache, stats)\n            node_id = next_node_id\n            next_node_id += 1\n            nodes_by_id[node_id] = {\n                "node_id": node_id,\n                "t": mid_t,\n                "z": refined_point[0],\n                "y": refined_point[1],\n                "x": refined_point[2],\n            }\n            inserted_ids.append(node_id)\n            current = nodes_by_id[node_id]\n            new_edges.append({\n                "source_id": previous_id,\n                "target_id": node_id,\n                "edge_prob": None,\n                "distance_um": edge_distance_um(nodes_by_id[previous_id], current),\n                "gap2_recovered": 1,\n            })\n            previous_id = node_id\n        new_edges.append({\n            "source_id": previous_id,\n            "target_id": start_id,\n            "edge_prob": None,\n            "distance_um": edge_distance_um(nodes_by_id[previous_id], target),\n            "gap2_recovered": 1,\n        })\n        stats["gap2_pairs_selected"] += 1\n        stats["gap2_added_nodes"] += len(inserted_ids)\n        stats["gap2_added_edges"] += 3\n\n    return nodes_by_id, [*edges, *new_edges]\n\n\ndef add_safe_divisions_postlink(\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n) -> list[dict[str, object]]:\n    if not OUTPUT_SAFE_DIVISIONS or not edges or not nodes_by_id:\n        return edges\n\n    out_by_source: dict[int, list[dict[str, object]]] = {}\n    incoming: set[int] = set()\n    for edge in edges:\n        out_by_source.setdefault(int(edge["source_id"]), []).append(edge)\n        incoming.add(int(edge["target_id"]))\n\n    ids_by_t: dict[int, list[int]] = {}\n    for node_id, node in nodes_by_id.items():\n        ids_by_t.setdefault(int(node["t"]), []).append(node_id)\n\n    existing_edges = {(int(edge["source_id"]), int(edge["target_id"])) for edge in edges}\n    global_cap = max(1, int(round(max(1, len(edges)) * SAFE_DIV_GLOBAL_FRAC_CAP)))\n    added: list[dict[str, object]] = []\n    used_targets: set[int] = set()\n\n    for t in sorted(ids_by_t):\n        child_frame_ids = ids_by_t.get(t + 1, [])\n        if not child_frame_ids:\n            continue\n        source_ids = [node_id for node_id in ids_by_t[t] if len(out_by_source.get(node_id, [])) == 1]\n        candidate_ids = [node_id for node_id in child_frame_ids if node_id not in incoming and node_id not in used_targets]\n        if not source_ids or not candidate_ids:\n            continue\n\n        frame_cap = max(1, int(round(len(source_ids) * SAFE_DIV_FRAME_FRAC_CAP)))\n        proposals: list[tuple[float, int, int, float, float]] = []\n        for source_id in source_ids:\n            source = nodes_by_id[source_id]\n            existing_child_edge = out_by_source[source_id][0]\n            existing_child_id = int(existing_child_edge["target_id"])\n            existing_child = nodes_by_id.get(existing_child_id)\n            if existing_child is None or int(existing_child["t"]) != t + 1:\n                continue\n            child_dist = edge_distance_um(source, existing_child)\n            if child_dist > SAFE_DIV_EXISTING_CHILD_MAX_UM:\n                continue\n            for candidate_id in candidate_ids:\n                if (source_id, candidate_id) in existing_edges:\n                    continue\n                candidate = nodes_by_id[candidate_id]\n                parent_dist = edge_distance_um(source, candidate)\n                if parent_dist > SAFE_DIV_MAX_UM:\n                    continue\n                sister_dist = edge_distance_um(existing_child, candidate)\n                if sister_dist > SAFE_DIV_SISTER_MAX_UM:\n                    continue\n                score = parent_dist + 0.15 * sister_dist\n                proposals.append((score, source_id, candidate_id, parent_dist, sister_dist))\n\n        stats["safe_division_candidates"] += len(proposals)\n        if not proposals:\n            continue\n        proposals.sort(key=lambda item: item[0])\n        added_this_frame = 0\n        for _, source_id, candidate_id, parent_dist, _ in proposals:\n            if len(added) >= global_cap:\n                stats["safe_division_skipped_cap"] += 1\n                break\n            if added_this_frame >= frame_cap:\n                break\n            if candidate_id in used_targets or candidate_id in incoming:\n                continue\n            candidate = nodes_by_id[candidate_id]\n            added.append({\n                "source_id": source_id,\n                "target_id": candidate_id,\n                "edge_prob": None,\n                "distance_um": parent_dist,\n                "safe_division": 1,\n            })\n            used_targets.add(candidate_id)\n            added_this_frame += 1\n\n    if added:\n        stats["safe_divisions_added"] = len(added)\n        return [*edges, *added]\n    return edges\n\n\ndef filter_short_track_components(\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n) -> tuple[dict[int, dict[str, object]], list[dict[str, object]]]:\n    if not OUTPUT_FILTER_SHORT_TRACKS or OUTPUT_MIN_TRACK_LEN <= 1 or not edges:\n        return nodes_by_id, edges\n\n    parent = {node_id: node_id for node_id in nodes_by_id}\n\n    def find(node_id: int) -> int:\n        while parent[node_id] != node_id:\n            parent[node_id] = parent[parent[node_id]]\n            node_id = parent[node_id]\n        return node_id\n\n    def union(a: int, b: int) -> None:\n        if a not in parent or b not in parent:\n            return\n        ra = find(a)\n        rb = find(b)\n        if ra != rb:\n            parent[ra] = rb\n\n    out_count: dict[int, int] = {}\n    for edge in edges:\n        source_id = int(edge["source_id"])\n        target_id = int(edge["target_id"])\n        union(source_id, target_id)\n        out_count[source_id] = out_count.get(source_id, 0) + 1\n\n    components: dict[int, list[int]] = {}\n    for node_id in nodes_by_id:\n        components.setdefault(find(node_id), []).append(node_id)\n\n    keep: set[int] = set()\n    for members in components.values():\n        has_division = any(out_count.get(node_id, 0) >= 2 for node_id in members)\n        if len(members) >= OUTPUT_MIN_TRACK_LEN or (OUTPUT_KEEP_DIVISION_COMPONENTS and has_division):\n            keep.update(members)\n\n    if not keep:\n        stats["short_track_filter_skipped_all"] += 1\n        return nodes_by_id, edges\n\n    removed_nodes = len(nodes_by_id) - len(keep)\n    if removed_nodes <= 0:\n        return nodes_by_id, edges\n\n    kept_nodes = {node_id: node for node_id, node in nodes_by_id.items() if node_id in keep}\n    kept_edges = [\n        edge for edge in edges\n        if int(edge["source_id"]) in kept_nodes and int(edge["target_id"]) in kept_nodes\n    ]\n    stats["short_track_components_removed"] = sum(1 for members in components.values() if not (set(members) & keep))\n    stats["short_track_nodes_removed"] = removed_nodes\n    stats["short_track_edges_removed"] = len(edges) - len(kept_edges)\n    return kept_nodes, kept_edges\n\n\ndef linefit_smooth_output_graph(\n    nodes_by_id: dict[int, dict[str, object]],\n    edges: list[dict[str, object]],\n    stats: dict[str, int],\n) -> dict[int, dict[str, object]]:\n    """Smooth linear track interiors without changing graph topology."""\n    if not OUTPUT_LINEFIT_SMOOTH or OUTPUT_LINEFIT_WEIGHT <= 0 or OUTPUT_LINEFIT_WINDOW <= 0 or not edges:\n        return nodes_by_id\n\n    predecessor: dict[int, list[int]] = {}\n    successor: dict[int, list[int]] = {}\n    for edge in edges:\n        source_id = int(edge["source_id"])\n        target_id = int(edge["target_id"])\n        source = nodes_by_id.get(source_id)\n        target = nodes_by_id.get(target_id)\n        if source is None or target is None:\n            continue\n        if int(target["t"]) != int(source["t"]) + 1:\n            continue\n        successor.setdefault(source_id, []).append(target_id)\n        predecessor.setdefault(target_id, []).append(source_id)\n\n    original_pos = {\n        node_id: np.array([float(node["z"]), float(node["y"]), float(node["x"])], dtype=np.float64)\n        for node_id, node in nodes_by_id.items()\n    }\n    updated_pos: dict[int, np.ndarray] = {}\n    weight = float(np.clip(OUTPUT_LINEFIT_WEIGHT, 0.0, 1.0))\n\n    for node_id in sorted(nodes_by_id):\n        neighbourhood: list[tuple[int, int]] = [(0, node_id)]\n\n        current = node_id\n        for step in range(1, OUTPUT_LINEFIT_WINDOW + 1):\n            prev_ids = predecessor.get(current, [])\n            if len(prev_ids) != 1:\n                break\n            current = prev_ids[0]\n            if current not in original_pos:\n                break\n            neighbourhood.append((-step, current))\n\n        current = node_id\n        for step in range(1, OUTPUT_LINEFIT_WINDOW + 1):\n            next_ids = successor.get(current, [])\n            if len(next_ids) != 1:\n                break\n            current = next_ids[0]\n            if current not in original_pos:\n                break\n            neighbourhood.append((step, current))\n\n        if len(neighbourhood) < 3:\n            stats["linefit_skipped_nodes"] += 1\n            continue\n\n        dts = np.array([delta for delta, _ in neighbourhood], dtype=np.float64)\n        coords = np.stack([original_pos[nid] for _, nid in neighbourhood])\n        fitted = np.array([np.polyval(np.polyfit(dts, coords[:, axis], 1), 0.0) for axis in range(3)], dtype=np.float64)\n        if not np.isfinite(fitted).all():\n            stats["linefit_skipped_nodes"] += 1\n            continue\n        updated_pos[node_id] = (1.0 - weight) * original_pos[node_id] + weight * fitted\n\n    for node_id, pos in updated_pos.items():\n        nodes_by_id[node_id]["z"] = float(pos[0])\n        nodes_by_id[node_id]["y"] = float(pos[1])\n        nodes_by_id[node_id]["x"] = float(pos[2])\n\n    stats["linefit_smoothed_nodes"] = len(updated_pos)\n    return nodes_by_id\n\n\ndef filter_output_graph(\n    nodes_by_id: dict[int, dict[str, object]],\n    raw_edges: list[dict[str, object]],\n    dataset: str | None = None,\n) -> tuple[dict[int, dict[str, object]], list[dict[str, object]], dict[str, int]]:\n    stats = {\n        "raw_edges": len(raw_edges),\n        "dropped_nonconsecutive_edges": 0,\n        "dropped_long_edges": 0,\n        "dropped_multi_parent_edges": 0,\n        "dropped_multi_child_edges": 0,\n        "dropped_division_edges": 0,\n        "gap_candidates": 0,\n        "gap_pairs_selected": 0,\n        "gap_reused_existing": 0,\n        "gap_inserted_synthetic": 0,\n        "gap_added_nodes": 0,\n        "gap_added_edges": 0,\n        "gap_skipped_node_cap": 0,\n        "gap_refined_synthetic": 0,\n        "gap_refine_failed": 0,\n        "gap_refine_rejected_shift": 0,\n        "pruned_isolated_nodes": 0,\n        "motion_relink_edges": 0,\n        "motion_relink_tight_edges": 0,\n        "motion_relink_relaxed_edges": 0,\n        "motion_relink_frames": 0,\n        "motion_relink_replaced_raw_edges": 0,\n        "motion_relink_fallback_raw": 0,\n        "motion_relink_skipped_large_frame": 0,\n        "gap2_candidates": 0,\n        "gap2_pairs_selected": 0,\n        "gap2_added_nodes": 0,\n        "gap2_added_edges": 0,\n        "gap2_skipped_cap": 0,\n        "safe_division_candidates": 0,\n        "safe_divisions_added": 0,\n        "safe_division_skipped_cap": 0,\n        "short_track_components_removed": 0,\n        "short_track_nodes_removed": 0,\n        "short_track_edges_removed": 0,\n        "short_track_filter_skipped_all": 0,\n        "linefit_smoothed_nodes": 0,\n        "linefit_skipped_nodes": 0,\n    }\n\n    edges: list[dict[str, object]] = []\n    for edge in raw_edges:\n        source = nodes_by_id.get(int(edge["source_id"]))\n        target = nodes_by_id.get(int(edge["target_id"]))\n        if source is None or target is None:\n            continue\n        if OUTPUT_ENFORCE_NEXT_FRAME and int(target["t"]) != int(source["t"]) + 1:\n            stats["dropped_nonconsecutive_edges"] += 1\n            continue\n        distance_um = edge_distance_um(source, target)\n        edge["distance_um"] = distance_um\n        if OUTPUT_EDGE_MAX_UM > 0 and distance_um > OUTPUT_EDGE_MAX_UM:\n            stats["dropped_long_edges"] += 1\n            continue\n        edges.append(edge)\n\n    if OUTPUT_MOTION_RELINK:\n        learned_edge_probs: dict[tuple[int, int], float] = {}\n        for edge in edges:\n            prob = edge.get("edge_prob")\n            if prob is None:\n                continue\n            try:\n                prob = float(prob)\n            except (TypeError, ValueError):\n                continue\n            if np.isfinite(prob):\n                key = (int(edge["source_id"]), int(edge["target_id"]))\n                learned_edge_probs[key] = max(learned_edge_probs.get(key, float("-inf")), prob)\n        motion_edges = motion_relink_edges(nodes_by_id, stats, learned_edge_probs)\n        if motion_edges:\n            stats["motion_relink_replaced_raw_edges"] = len(edges)\n            edges = motion_edges\n        else:\n            stats["motion_relink_fallback_raw"] = 1\n\n    if OUTPUT_SINGLE_PARENT_REPAIR and edges:\n        best_by_target: dict[int, dict[str, object]] = {}\n        for edge in edges:\n            target_id = int(edge["target_id"])\n            prev = best_by_target.get(target_id)\n            if prev is None or edge_sort_key(edge) > edge_sort_key(prev):\n                best_by_target[target_id] = edge\n        kept_ids = {id(edge) for edge in best_by_target.values()}\n        stats["dropped_multi_parent_edges"] = sum(1 for edge in edges if id(edge) not in kept_ids)\n        edges = [edge for edge in edges if id(edge) in kept_ids]\n\n    if OUTPUT_SINGLE_CHILD_REPAIR and edges:\n        best_by_source: dict[int, dict[str, object]] = {}\n        for edge in edges:\n            source_id = int(edge["source_id"])\n            prev = best_by_source.get(source_id)\n            if prev is None or edge_sort_key(edge) > edge_sort_key(prev):\n                best_by_source[source_id] = edge\n        kept_ids = {id(edge) for edge in best_by_source.values()}\n        stats["dropped_multi_child_edges"] = sum(1 for edge in edges if id(edge) not in kept_ids)\n        edges = [edge for edge in edges if id(edge) in kept_ids]\n\n    nodes_by_id, edges = close_single_frame_gaps(nodes_by_id, edges, stats, dataset=dataset)\n    nodes_by_id, edges = recover_strict_gap2(nodes_by_id, edges, stats, dataset=dataset)\n    edges = add_safe_divisions_postlink(nodes_by_id, edges, stats)\n\n    if OUTPUT_DIVISION_GEOMETRY_FILTER and edges:\n        by_source: dict[int, list[dict[str, object]]] = {}\n        for edge in edges:\n            by_source.setdefault(int(edge["source_id"]), []).append(edge)\n\n        filtered: list[dict[str, object]] = []\n        for source_id, source_edges in by_source.items():\n            if len(source_edges) <= 1:\n                filtered.extend(source_edges)\n                continue\n\n            ranked = sorted(source_edges, key=edge_sort_key, reverse=True)\n            source = nodes_by_id[source_id]\n            top1 = ranked[0]\n            top2 = ranked[1]\n            d1 = float(top1["distance_um"])\n            d2 = float(top2["distance_um"])\n            sister = edge_distance_um(nodes_by_id[int(top1["target_id"])], nodes_by_id[int(top2["target_id"])])\n            valid_division = (\n                max(d1, d2) <= DIV_PARENT_MAX_UM\n                and sister <= DIV_SISTER_MAX_UM\n                and int(nodes_by_id[int(top1["target_id"])] ["t"]) == int(source["t"]) + 1\n                and int(nodes_by_id[int(top2["target_id"])] ["t"]) == int(source["t"]) + 1\n            )\n            if valid_division:\n                filtered.extend([top1, top2])\n                stats["dropped_division_edges"] += max(0, len(ranked) - 2)\n            elif DIV_DROP_TO_SINGLE_IF_BAD:\n                filtered.append(top1)\n                stats["dropped_division_edges"] += len(ranked) - 1\n            else:\n                filtered.extend(ranked)\n        edges = filtered\n\n    if OUTPUT_PRUNE_ISOLATED:\n        incident = {int(edge["source_id"]) for edge in edges} | {int(edge["target_id"]) for edge in edges}\n        if incident:\n            kept_nodes = {node_id: node for node_id, node in nodes_by_id.items() if node_id in incident}\n            stats["pruned_isolated_nodes"] = len(nodes_by_id) - len(kept_nodes)\n            nodes_by_id = kept_nodes\n            edges = [edge for edge in edges if int(edge["source_id"]) in nodes_by_id and int(edge["target_id"]) in nodes_by_id]\n\n    nodes_by_id, edges = filter_short_track_components(nodes_by_id, edges, stats)\n    nodes_by_id = linefit_smooth_output_graph(nodes_by_id, edges, stats)\n\n    return nodes_by_id, edges, stats\n\n\nFULL_FRAME_DETECTOR = load_full_frame_detector()\n\ngeffs = sorted((REPO_DIR / "predictions").glob(f"*/{METHOD}/split_0/*.geff"))\nprint(f"Found {len(geffs)} prediction graphs")\nif len(geffs) != len(test_stems):\n    found = {path.stem for path in geffs}\n    missing = sorted(set(test_stems) - found)\n    raise RuntimeError(f"Expected {len(test_stems)} graphs, found {len(geffs)}. Missing: {missing[:10]}")\n\nstats_rows: list[dict[str, object]] = []\nseen_datasets: set[str] = set()\nrow_id = 0\ntotal_nodes = 0\ntotal_edges = 0\n\nwith SUBMISSION_PATH.open("w", newline="") as f:\n    writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)\n    writer.writeheader()\n\n    for geff_path in geffs:\n        dataset = geff_path.stem\n        seen_datasets.add(dataset)\n        graph = graph_from_geff(geff_path)\n\n        nodes_by_id: dict[int, dict[str, object]] = {}\n        for row in graph.node_attrs().iter_rows(named=True):\n            node_id = int(row["node_id"])\n            nodes_by_id[node_id] = {\n                "node_id": node_id,\n                "t": int(row["t"]),\n                "z": float(row["z"]),\n                "y": float(row["y"]),\n                "x": float(row["x"]),\n            }\n\n        raw_edges: list[dict[str, object]] = []\n        for row in graph.edge_attrs().iter_rows(named=True):\n            edge_prob = row.get("edge_prob") if hasattr(row, "get") else None\n            raw_edges.append({\n                "source_id": int(row["source_id"]),\n                "target_id": int(row["target_id"]),\n                "edge_prob": None if edge_prob is None else float(edge_prob),\n            })\n\n        raw_node_count = len(nodes_by_id)\n        full_frame_stats = fuse_full_frame_nodes(dataset, nodes_by_id, FULL_FRAME_DETECTOR)\n        fused_node_count = len(nodes_by_id)\n        nodes_by_id, edges, filter_stats = filter_output_graph(nodes_by_id, raw_edges, dataset=dataset)\n        if not nodes_by_id:\n            raise AssertionError(f"{dataset}: post-processing removed every node")\n\n        for node_id in sorted(nodes_by_id):\n            node = nodes_by_id[node_id]\n            writer.writerow({\n                "id": row_id,\n                "dataset": dataset,\n                "row_type": "node",\n                "node_id": int(node["node_id"]),\n                "t": int(node["t"]),\n                "z": int(round(float(node["z"]))),\n                "y": int(round(float(node["y"]))),\n                "x": int(round(float(node["x"]))),\n                "source_id": -1,\n                "target_id": -1,\n            })\n            row_id += 1\n\n        division_sources: dict[int, int] = {}\n        for edge in edges:\n            source_id = int(edge["source_id"])\n            target_id = int(edge["target_id"])\n            if source_id not in nodes_by_id or target_id not in nodes_by_id:\n                raise AssertionError(f"{dataset}: dangling edge after filtering")\n            writer.writerow({\n                "id": row_id,\n                "dataset": dataset,\n                "row_type": "edge",\n                "node_id": -1,\n                "t": -1,\n                "z": -1,\n                "y": -1,\n                "x": -1,\n                "source_id": source_id,\n                "target_id": target_id,\n            })\n            row_id += 1\n            division_sources[source_id] = division_sources.get(source_id, 0) + 1\n\n        node_count = len(nodes_by_id)\n        edge_count = len(edges)\n        total_nodes += node_count\n        total_edges += edge_count\n        stats_rows.append({\n            "dataset": dataset,\n            "raw_nodes": raw_node_count,\n            "raw_nodes_after_full_frame_fusion": fused_node_count,\n            "nodes": node_count,\n            "raw_edges": filter_stats["raw_edges"],\n            "edges": edge_count,\n            "division_like_sources": sum(1 for count in division_sources.values() if count >= 2),\n            "edge_to_node_ratio": edge_count / max(node_count, 1),\n            "gap_added_nodes_frac": filter_stats.get("gap_added_nodes", 0) / max(fused_node_count, 1),\n            **full_frame_stats,\n            **filter_stats,\n        })\n\nexpected_datasets = set(test_stems)\nmissing_datasets = sorted(expected_datasets - seen_datasets)\nextra_datasets = sorted(seen_datasets - expected_datasets)\nif missing_datasets or extra_datasets:\n    raise AssertionError({"missing": missing_datasets[:10], "extra": extra_datasets[:10]})\nassert row_id == total_nodes + total_edges, "Internal row counter mismatch"\nassert total_nodes > 0, "No node rows produced"\n\nheader = SUBMISSION_PATH.open().readline().strip().split(",")\nassert header == CSV_COLUMNS, f"Bad CSV header: {header}"\n\nstats = pd.DataFrame(stats_rows).sort_values("dataset").reset_index(drop=True)\nstats["predict_minutes_total"] = predict_seconds / 60.0\nstats["experiment_tag"] = EXPERIMENT_TAG\nstats.to_csv(RUN_STATS_PATH, index=False)\n\nprint(f"Wrote {SUBMISSION_PATH} with {row_id:,} rows")\nprint(f"Node rows: {total_nodes:,} | edge rows: {total_edges:,}")\nprint(f"Wrote {RUN_STATS_PATH}")\ndisplay(stats.describe(include="all"))\ndisplay(pd.read_csv(SUBMISSION_PATH, nrows=8))\n'
COPY_OUTPUT_CODE = "from pathlib import Path\nimport shutil\nimport pandas as pd\n\nsubmission_path = Path('submission.csv')\nrun_stats_path = Path('run_stats.csv')\n\nbatch_submission = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_submission.csv'\nbatch_stats = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_run_stats.csv'\nraw_backup = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_submission_raw_before_clip.csv'\nstable_submission = REPORT_DIR / 'deepcenter_local_validation_submission.csv'\nstable_stats = REPORT_DIR / 'deepcenter_local_validation_run_stats.csv'\n\nif not submission_path.exists():\n    raise FileNotFoundError(submission_path)\n\nshutil.copy2(submission_path, batch_submission)\nif not raw_backup.exists():\n    shutil.copy2(submission_path, raw_backup)\n\nsub = pd.read_csv(batch_submission)\nnodes = sub['row_type'].eq('node')\n\ndef count_bad_node_coords(df):\n    d = df.loc[df['row_type'].eq('node'), ['t', 'z', 'y', 'x']]\n    return int(((d['t'] < 0) | (d['t'] > 99) |\n                (d['z'] < 0) | (d['z'] > 63) |\n                (d['y'] < 0) | (d['y'] > 255) |\n                (d['x'] < 0) | (d['x'] > 255)).sum())\n\nbad_before = count_bad_node_coords(sub)\nsub.loc[nodes, 't'] = sub.loc[nodes, 't'].clip(0, 99)\nsub.loc[nodes, 'z'] = sub.loc[nodes, 'z'].clip(0, 63)\nsub.loc[nodes, 'y'] = sub.loc[nodes, 'y'].clip(0, 255)\nsub.loc[nodes, 'x'] = sub.loc[nodes, 'x'].clip(0, 255)\n\nint_cols = ['id', 'node_id', 't', 'z', 'y', 'x', 'source_id', 'target_id']\nsub[int_cols] = sub[int_cols].astype('int64')\nsub.to_csv(batch_submission, index=False)\nbad_after = count_bad_node_coords(sub)\n\nif run_stats_path.exists():\n    shutil.copy2(run_stats_path, batch_stats)\n\nif WRITE_STABLE_ALIAS:\n    shutil.copy2(batch_submission, stable_submission)\n    if batch_stats.exists():\n        shutil.copy2(batch_stats, stable_stats)\n\nprint('raw backup:', raw_backup)\nprint('bad node coords before clip:', bad_before)\nprint('bad node coords after clip:', bad_after)\nprint('targeted clipped submission:', batch_submission)\nprint('targeted run stats:', batch_stats if batch_stats.exists() else 'missing')\nprint('WRITE_STABLE_ALIAS:', WRITE_STABLE_ALIAS)\nprint('Now run the local sparse-GT score cell below.')\n"
SCORE_CODE = "from pathlib import Path\nimport json\nimport math\n\nimport numpy as np\nimport pandas as pd\nimport zarr\nfrom scipy.optimize import linear_sum_assignment\nfrom tqdm.auto import tqdm\n\nMATCH_MAX_DISTANCE_UM = 7.0\nVOXEL_SCALE_UM = np.array([1.625, 0.40625, 0.40625], dtype=float)\n\nsubmission_path = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_submission.csv'\nif not submission_path.exists():\n    local_submission = Path('submission.csv')\n    if local_submission.exists():\n        submission_path = local_submission\n    else:\n        raise FileNotFoundError('No targeted DeepCenter submission CSV found. Run build + copy cells first.')\n\nprint('loading submission:', submission_path)\nsubmission_df = pd.read_csv(submission_path)\nscore_sample_ids = sorted(submission_df['dataset'].astype(str).unique())\nvalidation_df = pd.DataFrame({'sample_id': score_sample_ids})\nvalidation_df['embryo_id'] = validation_df['sample_id'].str.split('_').str[0]\nvalidation_df['geff_path'] = validation_df['sample_id'].map(lambda s: BASE_DIR / 'train' / f'{s}.geff')\nvalidation_df['has_geff'] = validation_df['geff_path'].map(lambda p: p.exists())\n\nnode_rows = submission_df['row_type'].eq('node')\nbad_node_coords = int((\n    (submission_df.loc[node_rows, 't'] < 0) | (submission_df.loc[node_rows, 't'] > 99) |\n    (submission_df.loc[node_rows, 'z'] < 0) | (submission_df.loc[node_rows, 'z'] > 63) |\n    (submission_df.loc[node_rows, 'y'] < 0) | (submission_df.loc[node_rows, 'y'] > 255) |\n    (submission_df.loc[node_rows, 'x'] < 0) | (submission_df.loc[node_rows, 'x'] > 255)\n).sum())\n\nprint('score samples:', len(validation_df))\nprint('submission rows:', len(submission_df))\nprint('node rows:', int(node_rows.sum()))\nprint('edge rows:', int((submission_df['row_type'] == 'edge').sum()))\nprint('bad node coordinate rows:', bad_node_coords)\ndisplay(validation_df[['sample_id', 'embryo_id', 'has_geff']].head(20))\nassert validation_df['has_geff'].all(), 'Some validation GEFF files are missing'\nassert bad_node_coords == 0, 'Submission still has out-of-bound node coordinates. Run the copy/clip cell first.'\n\nsubmission_groups = submission_df.groupby('dataset', sort=False)\n\n\ndef read_geff(geff_path: Path):\n    root = zarr.open(str(geff_path), mode='r')\n    node_ids = np.asarray(root['nodes/ids'][:]).astype(int)\n    nodes = pd.DataFrame({\n        'node_id': node_ids,\n        't': np.asarray(root['nodes/props/t/values'][:]).astype(int),\n        'z': np.asarray(root['nodes/props/z/values'][:]).astype(float),\n        'y': np.asarray(root['nodes/props/y/values'][:]).astype(float),\n        'x': np.asarray(root['nodes/props/x/values'][:]).astype(float),\n    })\n    edge_ids = np.asarray(root['edges/ids'][:]).astype(int)\n    edges = pd.DataFrame(edge_ids, columns=['source_id', 'target_id']) if edge_ids.size else pd.DataFrame(columns=['source_id', 'target_id'])\n\n    estimated = np.nan\n    meta_path = geff_path / 'zarr.json'\n    if meta_path.exists():\n        try:\n            meta = json.loads(meta_path.read_text())\n            attrs = meta.get('attributes', {})\n            estimated = attrs.get('estimated_number_of_nodes', np.nan)\n        except Exception:\n            pass\n    return nodes, edges, estimated\n\n\ndef physical_distance_matrix_um(a_zyx, b_zyx):\n    a = np.asarray(a_zyx, dtype=float) * VOXEL_SCALE_UM\n    b = np.asarray(b_zyx, dtype=float) * VOXEL_SCALE_UM\n    diff = a[:, None, :] - b[None, :, :]\n    return np.sqrt((diff * diff).sum(axis=2))\n\n\ndef match_pred_to_gt_by_time(pred_nodes, gt_nodes, max_distance_um=MATCH_MAX_DISTANCE_UM):\n    rows = []\n    pred_to_gt = {}\n    gt_to_pred = {}\n    common_times = sorted(set(pred_nodes['t']).intersection(set(gt_nodes['t'])))\n    for t in common_times:\n        pred_t = pred_nodes[pred_nodes['t'] == t].reset_index(drop=True)\n        gt_t = gt_nodes[gt_nodes['t'] == t].reset_index(drop=True)\n        if pred_t.empty or gt_t.empty:\n            continue\n        dist = physical_distance_matrix_um(pred_t[['z', 'y', 'x']].to_numpy(), gt_t[['z', 'y', 'x']].to_numpy())\n        rr, cc = linear_sum_assignment(dist)\n        for r, c in zip(rr, cc):\n            d = float(dist[r, c])\n            if d <= max_distance_um:\n                pred_node_id = int(pred_t.loc[r, 'node_id'])\n                gt_node_id = int(gt_t.loc[c, 'node_id'])\n                rows.append({'t': int(t), 'pred_node_id': pred_node_id, 'gt_node_id': gt_node_id, 'distance_um': d})\n                pred_to_gt[pred_node_id] = gt_node_id\n                gt_to_pred[gt_node_id] = pred_node_id\n    return pd.DataFrame(rows), pred_to_gt, gt_to_pred\n\n\ndef split_submission_group(sample_df):\n    nodes = sample_df[sample_df['row_type'] == 'node'][['node_id', 't', 'z', 'y', 'x']].copy()\n    edges = sample_df[sample_df['row_type'] == 'edge'][['source_id', 'target_id']].copy()\n    return nodes, edges\n\n\ndef division_diagnostics(pred_edges, gt_edges, pred_to_gt):\n    if len(gt_edges):\n        gt_out = gt_edges['source_id'].astype(int).value_counts()\n        gt_div_nodes = set(gt_out[gt_out >= 2].index.astype(int))\n    else:\n        gt_div_nodes = set()\n\n    if len(pred_edges):\n        pred_out = pred_edges['source_id'].astype(int).value_counts()\n        pred_div_nodes = set(pred_out[pred_out >= 2].index.astype(int))\n    else:\n        pred_div_nodes = set()\n\n    matched_pred_div_gt = {pred_to_gt[p] for p in pred_div_nodes if p in pred_to_gt}\n    direct_tp = len(gt_div_nodes & matched_pred_div_gt)\n    direct_fp = max(0, len(pred_div_nodes) - direct_tp)\n    direct_fn = max(0, len(gt_div_nodes) - direct_tp)\n    direct_j = direct_tp / max(1, direct_tp + direct_fp + direct_fn)\n\n    return {\n        'gt_divisions_direct': int(len(gt_div_nodes)),\n        'pred_divisions_direct': int(len(pred_div_nodes)),\n        'division_tp_direct_approx': int(direct_tp),\n        'division_fp_direct_approx': int(direct_fp),\n        'division_fn_direct_approx': int(direct_fn),\n        'division_jaccard_direct_approx': float(direct_j),\n    }\n\n\ndef score_one_sample(pred_nodes, pred_edges, gt_nodes, gt_edges, estimated_number_of_nodes=np.nan):\n    matches, pred_to_gt, gt_to_pred = match_pred_to_gt_by_time(pred_nodes, gt_nodes)\n    gt_edge_set = set(map(tuple, gt_edges[['source_id', 'target_id']].astype(int).to_numpy())) if len(gt_edges) else set()\n\n    edge_tp = 0\n    matched_pred_edges = 0\n    for row in pred_edges[['source_id', 'target_id']].astype(int).itertuples(index=False):\n        src_gt = pred_to_gt.get(int(row.source_id))\n        tgt_gt = pred_to_gt.get(int(row.target_id))\n        if src_gt is None or tgt_gt is None:\n            continue\n        matched_pred_edges += 1\n        if (src_gt, tgt_gt) in gt_edge_set:\n            edge_tp += 1\n\n    edge_fp_matched = max(0, matched_pred_edges - edge_tp)\n    edge_fn = max(0, len(gt_edge_set) - edge_tp)\n    edge_den_matched = max(1, edge_tp + edge_fp_matched + edge_fn)\n    edge_jaccard = edge_tp / edge_den_matched\n    edge_jaccard_naive = edge_tp / max(1, edge_tp + max(0, len(pred_edges) - edge_tp) + edge_fn)\n\n    if np.isfinite(estimated_number_of_nodes) and estimated_number_of_nodes > 0:\n        over_factor = min(1.0, float(estimated_number_of_nodes) / max(1, len(pred_nodes)))\n    else:\n        over_factor = 1.0\n\n    result = {\n        'pred_nodes': int(len(pred_nodes)),\n        'gt_nodes_sparse': int(len(gt_nodes)),\n        'matched_nodes': int(len(matches)),\n        'node_recall_sparse_gt': float(len(matches) / max(1, len(gt_nodes))),\n        'node_precision_sparse_gt': float(len(matches) / max(1, len(pred_nodes))),\n        'pred_edges': int(len(pred_edges)),\n        'gt_edges_sparse': int(len(gt_edge_set)),\n        'matched_pred_edges': int(matched_pred_edges),\n        'edge_tp': int(edge_tp),\n        'edge_fp_matched': int(edge_fp_matched),\n        'edge_fn': int(edge_fn),\n        'edge_den_matched': int(edge_den_matched),\n        'edge_jaccard_matched': float(edge_jaccard),\n        'edge_jaccard_naive': float(edge_jaccard_naive),\n        'estimated_number_of_nodes': float(estimated_number_of_nodes) if np.isfinite(estimated_number_of_nodes) else np.nan,\n        'node_overprediction_factor': float(over_factor),\n        'edge_jaccard_adjusted_approx': float(edge_jaccard * over_factor),\n        'match_distance_median_um': float(matches['distance_um'].median()) if len(matches) else np.nan,\n        'match_distance_p95_um': float(matches['distance_um'].quantile(0.95)) if len(matches) else np.nan,\n    }\n    result.update(division_diagnostics(pred_edges, gt_edges, pred_to_gt))\n    return result\n\nscore_rows = []\nfor row in tqdm(validation_df.itertuples(index=False), total=len(validation_df), desc='local sparse-GT score'):\n    sample_id = row.sample_id\n    gt_nodes, gt_edges, estimated = read_geff(Path(row.geff_path))\n    try:\n        sample_submission = submission_groups.get_group(sample_id)\n    except KeyError:\n        print('missing predictions for', sample_id)\n        continue\n    pred_nodes, pred_edges = split_submission_group(sample_submission)\n    if pred_nodes.empty:\n        print('missing node predictions for', sample_id)\n        continue\n    score = score_one_sample(pred_nodes, pred_edges, gt_nodes, gt_edges, estimated)\n    score.update({'method': 'deepcenter', 'sample_id': sample_id, 'embryo_id': row.embryo_id})\n    score_rows.append(score)\n\nscores_df = pd.DataFrame(score_rows)\nif scores_df.empty:\n    raise RuntimeError('No scores were computed.')\n\nweighted_den = scores_df['edge_den_matched'].clip(lower=1)\nweighted_edge = float((scores_df['edge_jaccard_matched'] * weighted_den).sum() / weighted_den.sum())\nweighted_adjusted = float((scores_df['edge_jaccard_adjusted_approx'] * weighted_den).sum() / weighted_den.sum())\n\nsummary_rows = [{\n    'method': 'deepcenter',\n    'n_samples': int(len(scores_df)),\n    'local_weighted_edge_jaccard_approx': weighted_edge,\n    'local_weighted_adjusted_edge_jaccard_approx': weighted_adjusted,\n    'mean_edge_jaccard_matched': float(scores_df['edge_jaccard_matched'].mean()),\n    'median_edge_jaccard_matched': float(scores_df['edge_jaccard_matched'].median()),\n    'p10_edge_jaccard_matched': float(scores_df['edge_jaccard_matched'].quantile(0.10)),\n    'min_edge_jaccard_matched': float(scores_df['edge_jaccard_matched'].min()),\n    'mean_node_recall_sparse_gt': float(scores_df['node_recall_sparse_gt'].mean()),\n    'total_edge_tp': int(scores_df['edge_tp'].sum()),\n    'total_edge_fp_matched': int(scores_df['edge_fp_matched'].sum()),\n    'total_edge_fn': int(scores_df['edge_fn'].sum()),\n    'total_gt_divisions_direct': int(scores_df['gt_divisions_direct'].sum()),\n    'total_pred_divisions_direct': int(scores_df['pred_divisions_direct'].sum()),\n    'direct_division_jaccard_diagnostic': float(\n        scores_df['division_tp_direct_approx'].sum() /\n        max(1, scores_df['division_tp_direct_approx'].sum() + scores_df['division_fp_direct_approx'].sum() + scores_df['division_fn_direct_approx'].sum())\n    ),\n}]\nsummary_df = pd.DataFrame(summary_rows)\nby_embryo_df = scores_df.groupby('embryo_id').agg(\n    n_samples=('sample_id', 'count'),\n    mean_edge_jaccard_matched=('edge_jaccard_matched', 'mean'),\n    median_edge_jaccard_matched=('edge_jaccard_matched', 'median'),\n    min_edge_jaccard_matched=('edge_jaccard_matched', 'min'),\n    mean_node_recall_sparse_gt=('node_recall_sparse_gt', 'mean'),\n    mean_pred_nodes=('pred_nodes', 'mean'),\n    mean_pred_edges=('pred_edges', 'mean'),\n).reset_index()\nworst_df = scores_df.sort_values('edge_jaccard_matched').head(25).copy()\n\nscore_path = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_scores.csv'\nsummary_path = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_score_summary.csv'\nworst_path = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_worst_samples.csv'\nby_embryo_path = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_by_embryo.csv'\nstable_score_path = REPORT_DIR / 'deepcenter_local_validation_scores.csv'\nstable_summary_path = REPORT_DIR / 'deepcenter_local_validation_score_summary.csv'\nstable_worst_path = REPORT_DIR / 'deepcenter_local_validation_worst_samples.csv'\n\nscores_df.to_csv(score_path, index=False)\nsummary_df.to_csv(summary_path, index=False)\nworst_df.to_csv(worst_path, index=False)\nby_embryo_df.to_csv(by_embryo_path, index=False)\nscores_df.to_csv(stable_score_path, index=False)\nsummary_df.to_csv(stable_summary_path, index=False)\nworst_df.to_csv(stable_worst_path, index=False)\n\nprint('LOCAL SCORE SUMMARY')\ndisplay(summary_df)\nprint('BY EMBRYO')\ndisplay(by_embryo_df)\nprint('PER-SAMPLE SCORES')\ndisplay(scores_df)\nprint('WORST SAMPLES')\ndisplay(worst_df)\n\nprint('saved scores:', score_path)\nprint('saved summary:', summary_path)\nprint('saved worst:', worst_path)\nprint('stable summary:', stable_summary_path)\nprint('Note: direct division diagnostic is intentionally conservative and is not the official Kaggle division metric.')\n"

# Build repo/config once, install/import dependencies once, and restore checkpoint predictions once.
set_active_variant(VARIANTS_TO_RUN[0])
exec(CONFIG_CODE, globals())
exec(DEPENDENCY_CODE, globals())
exec(RESTORE_CODE, globals())
print('Preparation complete. Restored selected checkpoint graphs once; variants will reuse them.')

## Run Variant Ablation

This cell loops over `VARIANTS_TO_RUN`, rebuilds the submission CSV for each variant, clips coordinates, scores against sparse GT, and writes a combined summary CSV.

Progress bars show the outer variant loop and the four big stages for each variant: config, build CSV, copy/clip, and local score. The build and score cells also keep their own detailed output/progress where available.

In [ ]:
from pathlib import Path
import pandas as pd
import time
from tqdm.auto import tqdm
from IPython.display import clear_output

ablation_rows = []
start_all = time.time()

variant_bar = tqdm(VARIANTS_TO_RUN, desc='postprocess variants', unit='variant')
for i, variant_name in enumerate(variant_bar, start=1):
    variant_bar.set_postfix_str(variant_name)
    set_active_variant(variant_name)
    variant_start = time.time()

    stage_bar = tqdm(['config', 'build_csv', 'copy_clip', 'score'], desc=f'{variant_name} stages', leave=False)

    # Recompute the public-notebook config after changing env overrides.
    stage_bar.set_postfix_str('reading env/config')
    exec(CONFIG_CODE, globals())
    stage_bar.update(1)

    stage_bar.set_postfix_str('building submission.csv')
    exec(BUILD_SUBMISSION_CODE, globals())
    stage_bar.update(1)

    stage_bar.set_postfix_str('copying + clipping coords')
    exec(COPY_OUTPUT_CODE, globals())
    stage_bar.update(1)

    stage_bar.set_postfix_str('local sparse-GT score')
    exec(SCORE_CODE, globals())
    stage_bar.update(1)
    stage_bar.close()

    summary_path = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_score_summary.csv'
    scores_path = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_scores.csv'
    stats_path = REPORT_DIR / f'{OUTPUT_PREFIX}_local_validation_run_stats.csv'

    summary = pd.read_csv(summary_path).iloc[0].to_dict()
    scores = pd.read_csv(scores_path)
    stats = pd.read_csv(stats_path) if stats_path.exists() else pd.DataFrame()

    row = {
        'variant': variant_name,
        'variant_index': i,
        'target_set': TARGET_SET_NAME,
        'n_samples': int(summary.get('n_samples', len(scores))),
        'weighted_edge': float(summary.get('local_weighted_edge_jaccard_approx', float('nan'))),
        'weighted_adjusted': float(summary.get('local_weighted_adjusted_edge_jaccard_approx', float('nan'))),
        'mean_edge': float(summary.get('mean_edge_jaccard_matched', float('nan'))),
        'median_edge': float(summary.get('median_edge_jaccard_matched', float('nan'))),
        'p10_edge': float(summary.get('p10_edge_jaccard_matched', float('nan'))),
        'min_edge': float(summary.get('min_edge_jaccard_matched', float('nan'))),
        'mean_node_recall': float(summary.get('mean_node_recall_sparse_gt', float('nan'))),
        'edge_tp': int(summary.get('total_edge_tp', scores['edge_tp'].sum())),
        'edge_fp': int(summary.get('total_edge_fp_matched', scores['edge_fp_matched'].sum())),
        'edge_fn': int(summary.get('total_edge_fn', scores['edge_fn'].sum())),
        'gt_divisions_direct': int(summary.get('total_gt_divisions_direct', scores.get('gt_divisions_direct', pd.Series(dtype=int)).sum())),
        'pred_divisions_direct': int(summary.get('total_pred_divisions_direct', scores.get('pred_divisions_direct', pd.Series(dtype=int)).sum())),
        'elapsed_min': round((time.time() - variant_start) / 60, 2),
    }
    if len(stats):
        for col in [
            'safe_divisions_added', 'division_like_sources', 'motion_relink_relaxed_edges',
            'gap_added_nodes', 'short_track_nodes_removed', 'dropped_long_edges',
            'linefit_smoothed_nodes', 'linefit_skipped_nodes',
        ]:
            if col in stats.columns:
                row[col] = int(stats[col].sum())

    ablation_rows.append(row)
    progress = pd.DataFrame(ablation_rows)
    baseline = progress.iloc[0]
    progress['delta_weighted_vs_baseline'] = progress['weighted_edge'] - float(baseline['weighted_edge'])
    progress['delta_tp_vs_baseline'] = progress['edge_tp'] - int(baseline['edge_tp'])
    progress['delta_fn_vs_baseline'] = progress['edge_fn'] - int(baseline['edge_fn'])
    display(progress.sort_values('weighted_edge', ascending=False))

ablation_df = pd.DataFrame(ablation_rows)
baseline = ablation_df.iloc[0]
ablation_df['delta_weighted_vs_baseline'] = ablation_df['weighted_edge'] - float(baseline['weighted_edge'])
ablation_df['delta_mean_vs_baseline'] = ablation_df['mean_edge'] - float(baseline['mean_edge'])
ablation_df['delta_tp_vs_baseline'] = ablation_df['edge_tp'] - int(baseline['edge_tp'])
ablation_df['delta_fp_vs_baseline'] = ablation_df['edge_fp'] - int(baseline['edge_fp'])
ablation_df['delta_fn_vs_baseline'] = ablation_df['edge_fn'] - int(baseline['edge_fn'])

summary_out = REPORT_DIR / f'deepcenter_ablation_{TARGET_SET_NAME}_summary.csv'
ablation_df.to_csv(summary_out, index=False)
print('saved ablation summary:', summary_out)
print('total elapsed minutes:', round((time.time() - start_all) / 60, 2))
display(ablation_df.sort_values('weighted_edge', ascending=False))

## Final Ranking

Use this after the loop, or rerun it later, to rank variants from the saved summary.

In [ ]:
from pathlib import Path
import pandas as pd

summary_path = REPORT_DIR / f'deepcenter_ablation_{TARGET_SET_NAME}_summary.csv'
if not summary_path.exists():
    raise FileNotFoundError(summary_path)

ablation_df = pd.read_csv(summary_path)
base = ablation_df.iloc[0]
cols = [
    'variant', 'weighted_edge', 'delta_weighted_vs_baseline', 'mean_edge', 'median_edge',
    'p10_edge', 'min_edge', 'mean_node_recall', 'edge_tp', 'edge_fp', 'edge_fn',
    'delta_tp_vs_baseline', 'delta_fp_vs_baseline', 'delta_fn_vs_baseline',
    'pred_divisions_direct', 'safe_divisions_added', 'motion_relink_relaxed_edges',
    'gap_added_nodes', 'short_track_nodes_removed', 'elapsed_min',
]
cols = [c for c in cols if c in ablation_df.columns]
ranked = ablation_df.sort_values(['weighted_edge', 'mean_node_recall'], ascending=False)[cols]
display(ranked)

best = ranked.iloc[0]['variant']
print('Best local targeted variant:', best)
print('Baseline weighted:', round(float(base['weighted_edge']), 6))
print('Best weighted:', round(float(ranked.iloc[0]['weighted_edge']), 6))
print('Delta:', round(float(ranked.iloc[0].get('delta_weighted_vs_baseline', 0.0)), 6))

print('\nDecision rule reminder: promote only if weighted edge improves, guard samples do not collapse, FP does not spike, and FN/TP direction is favorable.')